# **Proyecto Integrador**
# ***LUMINA* - La Biblia Viva con IA**

Plataforma conversacional que facilita la comprensión y aplicación contextual del texto bíblico mediante NLP y RAG, con un enfoque responsable, trazable y centrado en el usuario.

---

> *"La explicación de tus palabras nos da luz; y da entendimiento a los de mente sencilla."*
>
> — Salmos 119:130

---

### **Avance 4: Modelos alternativos**
**Equipo:** *#19*  
**Curso:** *Proyecto Integrador*  
**Posgrado:** *MNA - Maestría en Inteligencia Artificial Aplicada*  
**Institución:** Tecnológico de Monterrey  
**Fecha:** *22 de febrero del 2026*  

---

### **Miembros del equipo**
- **A01732505** - Steven Sebastian Brutscher Cortez
- **A01795323** - Anghelo Daniel Pérez Martínez
- **A01423059** - Esmeralda González García

---
> *Este notebook corresponde al **Avance 4: Modelos Alternativos** del curso Proyecto Integrador de la Maestría en Inteligencia Artificial Aplicada (MNA) del Tecnológico de Monterrey*

De acuerdo con **CRISP-ML(Q)**, este trabajo se ubica principalmente en la fase de:

- **Modeling**: En esta fase se exploran múltiples configuraciones de modelos relevantes para el problema, se comparan formalmente sus métricas de desempeño, se ajustan sus hiperparámetros y se selecciona el modelo individual final óptimo.

Asimismo, este avance también impacta parcialmente las fases de:

- **Evaluation**, mediante comparación cuantitativa multi-métrica.
- **Deployment readiness**, al analizar latencia, estabilidad y costo.

Este notebook continúa el pipeline desarrollado en los avances anteriores (EDA y Feature Engineering), utilizando los artefactos ya construidos para evaluar modelos alternativos dentro de la arquitectura RAG del sistema LUMINA.


---

> *“Porque las cosas invisibles de él, su eterno poder y deidad, se hacen claramente visibles desde la creación del mundo, siendo entendidas por medio de las cosas hechas…”*  
>
> Romanos 1:20

---

# **Introducción**

En este avance se construyen múltiples configuraciones experimentales del sistema LUMINA con el objetivo de identificar cuál proporciona el mejor desempeño en la tarea de orientación pastoral basada en Escritura.

A diferencia del Avance 3, donde se estableció un baseline formal comparando enfoques LLM-only y RAG (Dense y Hybrid), en este entregable se explora una gama más diversa de modelos individuales (no ensambles), variando:

- Algoritmos de recuperación (BM25, Dense, Hybrid)
- Representaciones semánticas (SentenceTransformers vs OpenAI embeddings)
- Arquitectura del pipeline (con y sin reranking)
- Configuraciones de hiperparámetros (k, α, temperature, etc.)

El objetivo no es únicamente encontrar el modelo con mayor puntuación promedio, sino comprender los trade-offs entre:
- Calidad generativa
- Fidelidad bíblica (grounding)
- Métricas de Information Retrieval
- Latencia
- Complejidad del sistema
- Costos computacionales

Este análisis permitirá seleccionar un modelo individual final, justificando la decisión desde una perspectiva técnica y de negocio.

# **Objetivos de aprendizaje**

Al finalizar este avance se espera:

1. Explorar múltiples configuraciones de modelos individuales relevantes para un sistema RAG pastoral.
2. Evaluar comparativamente su desempeño utilizando métricas generativas e IR.
3. Diseñar experimentos reproducibles y estructurados.
4. Realizar ajuste fino de hiperparámetros en los modelos con mejor desempeño.
5. Seleccionar y justificar formalmente el modelo final considerando métricas, interpretabilidad y costo computacional.

# **Relación con el Avance 3 – Baseline**

En el Avance 3 se estableció una línea base experimental comparando:

- LLM-only
- Dense RAG
- Hybrid RAG

Se concluyó que:
- Dense Retrieval mostró mejor balance entre rendimiento y simplicidad.
- k = 3 fue un valor óptimo.
- Hybrid no aportó mejoras sustanciales bajo configuración inicial.

Este avance parte de esos hallazgos y amplía el espacio experimental, introduciendo nuevos modelos alternativos y ajustes de hiperparámetros más sistemáticos.

El Baseline no se descarta: se incorpora como uno de los modelos comparativos dentro del nuevo marco experimental.

# **Conexión con CRISP-ML(Q)**

Este avance corresponde principalmente a las fases:

- **Modeling** → Construcción de múltiples modelos alternativos.
- **Evaluation** → Comparación estructurada mediante métricas cuantitativas.
- **Quality Assurance** → Validación de fidelidad bíblica y detección de citas fuera de contexto.

CRISP-ML(Q) enfatiza:
- Reproducibilidad
- Trazabilidad de configuraciones
- Evaluación sistemática
- Documentación de decisiones

Por ello, cada modelo será definido mediante configuraciones explícitas y los resultados serán almacenados para permitir replicabilidad completa dentro del repositorio.


---

> *"Si alguno de ustedes quiere construir una torre, ¿no se sienta primero a calcular los gastos y ver si tiene lo suficiente para terminarla?"*
>
> — Lucas 14:28

---

# **PARTE 1 - Setup y definición de rutas**

En esta sección se:

1. Monta Google Drive.
2. Define la ruta base del proyecto para el Avance 4.
3. Declaran rutas explícitas a:
   - Corpus tabular (.parquet)
   - Embeddings (.npz)
   - Manifest (.json)
4. Se fijan semillas de reproducibilidad.
5. Se cargan librerías necesarias para experimentación.

No se utilizará detección automática de rutas.
Todas las rutas serán explícitas para evitar ambigüedades.

In [1]:
# ============================================================
# Montando Drive
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================
# 1. Librerías básicas
# ============================

import os
import json
import time
import random
import numpy as np
import pandas as pd

from pathlib import Path

# Instalación de dependencias (si es necesario)
!pip install -q rank_bm25
!pip install -q sentence-transformers
print("Dependencias instaladas correctamente.")

# Information Retrieval
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Control de warnings
import warnings
warnings.filterwarnings("ignore")

# ============================
# 2. Reproducibilidad
# ============================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Semilla global fijada en: {SEED}")

# ============================
# 3. Definir rutas explícitas
# ============================

BASE_PATH = Path(
    "/content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/"
    "Avance 4 - Modelos alternativos/Avance 4 - Steven/"
)

PARQUET_PATH = BASE_PATH / "lumina_chunks_tabular_20260207_084138.parquet"
EMBEDDINGS_PATH = BASE_PATH / "lumina_embeddings_20260207_084138.npz"
MANIFEST_PATH = BASE_PATH / "embedding_manifest_20260207_084138.json"

BIBLE_STRUCTURED_PATH = BASE_PATH / "bible_structured.csv"

print("Rutas definidas correctamente:")
print("PARQUET:", PARQUET_PATH)
print("EMBEDDINGS:", EMBEDDINGS_PATH)
print("MANIFEST:", MANIFEST_PATH)
print("BIBLE_STRUCTURED:", BIBLE_STRUCTURED_PATH)

# ============================
# 4. Verificación de existencia
# ============================

assert PARQUET_PATH.exists(), "No se encontró el archivo parquet."
assert EMBEDDINGS_PATH.exists(), "No se encontró el archivo embeddings."
assert MANIFEST_PATH.exists(), "No se encontró el archivo manifest."
assert BIBLE_STRUCTURED_PATH.exists(), "No se encontró bible_structured.csv."

print("\nTodos los archivos fueron encontrados correctamente.")

Dependencias instaladas correctamente.
Semilla global fijada en: 42
Rutas definidas correctamente:
PARQUET: /content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/Avance 4 - Modelos alternativos/Avance 4 - Steven/lumina_chunks_tabular_20260207_084138.parquet
EMBEDDINGS: /content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/Avance 4 - Modelos alternativos/Avance 4 - Steven/lumina_embeddings_20260207_084138.npz
MANIFEST: /content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/Avance 4 - Modelos alternativos/Avance 4 - Steven/embedding_manifest_20260207_084138.json
BIBLE_STRUCTURED: /content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/Avance 4 - Modelos alternativos/Avance 4 - Steven/bible_structured.csv

Todos los archivos fueron encontrados correctamente.


### Interpretación (Parte 1)

- Se fijó una semilla global para garantizar reproducibilidad.
- Se montó Google Drive correctamente.
- Las rutas fueron definidas de manera explícita.
- Todos los artefactos requeridos existen en la carpeta de trabajo del Avance 4.

Con esto se garantiza que el experimento es trazable y que no depende de rutas heredadas de entregables anteriores.

---

> *"Y me buscaréis y hallaréis, porque me buscaréis de todo vuestro corazón."*
>
> — Jeremías 29:13

---

# **PARTE 2 – Carga y validación de artefactos**

En esta sección se cargan y validan los artefactos principales del proyecto, generados en el Avance 2:

- **Corpus tabular** (`.parquet`) con los chunks bíblicos y sus metadatos.
- **Embeddings Dense** (`.npz`) generados con SentenceTransformers.
- **Manifest** (`.json`) con metadata del modelo de embeddings.

Además, se verifican explícitamente:

- Dimensiones reales del corpus y embeddings.
- Presencia de columnas clave (`chunk_id`, `chunk_ref`, `chunk_text_rv1909`).
- Consistencia 1:1 entre filas del parquet y embeddings.
- Confirmación de si los embeddings están normalizados (para cosine similarity).

Esta validación es crítica para asegurar reproducibilidad y evitar errores silenciosos en la evaluación de modelos.

In [3]:
# ==========================================
# Parte 2 - Carga y validación de artefactos
# ==========================================

# ----------------------------
# 1) Parámetros / columnas clave
# ----------------------------
ID_COL = "chunk_id"
REF_COL = "chunk_ref"
TEXT_COL = "chunk_text_rv1909"  # decisión técnica confirmada

REQUIRED_COLS = [ID_COL, REF_COL, TEXT_COL]

print("Columnas clave definidas:")
print(f"ID_COL   = {ID_COL}")
print(f"REF_COL  = {REF_COL}")
print(f"TEXT_COL = {TEXT_COL}")

# ----------------------------
# 2) Cargar parquet (chunks tabulares)
# ----------------------------
t0 = time.time()
df_chunks = pd.read_parquet(PARQUET_PATH)
t_parquet = time.time() - t0

print("\nParquet cargado.")
print(f"Tiempo de carga parquet: {t_parquet:.3f} s")
print(f"Shape df_chunks: {df_chunks.shape}")

# Vista rápida
display(df_chunks.head(3))

# ----------------------------
# 3) Validar columnas mínimas
# ----------------------------
missing_cols = [c for c in REQUIRED_COLS if c not in df_chunks.columns]
assert len(missing_cols) == 0, f"Faltan columnas requeridas en df_chunks: {missing_cols}"

print("\nColumnas requeridas presentes:", REQUIRED_COLS)

# Validar nulos en columnas críticas (no debe haber nulos)
for c in REQUIRED_COLS:
    n_null = df_chunks[c].isna().sum()
    assert n_null == 0, f"La columna '{c}' tiene {n_null} valores nulos."
print("No hay valores nulos en columnas críticas.")

# Validar unicidad del ID (chunk_id debe ser único)
n_unique = df_chunks[ID_COL].nunique()
assert n_unique == len(df_chunks), f"{ID_COL} no es único: {n_unique} únicos vs {len(df_chunks)} filas."
print(f"{ID_COL} es único (n={n_unique}).")

# ----------------------------
# 4) Cargar embeddings (.npz) - versión real (multi-key)
# ----------------------------
t0 = time.time()
npz = np.load(EMBEDDINGS_PATH, allow_pickle=True)
t_npz = time.time() - t0

print("\nNPZ cargado.")
print(f"Tiempo de carga embeddings: {t_npz:.3f} s")
print(f"Keys disponibles en npz: {list(npz.files)}")

# ----------------------------
# 5) Seleccionar embedding correcto según TEXT_COL
# ----------------------------
# Decisión técnica: usamos RV1909 como texto principal
EMB_KEY = "emb_rv1909" if TEXT_COL == "chunk_text_rv1909" else "emb_vbl"

assert EMB_KEY in npz.files, f"No existe la key esperada '{EMB_KEY}' en el npz."

embeddings = npz[EMB_KEY]
npz_chunk_ids = npz["chunk_id"]

print(f"\nEmbedding seleccionado: {EMB_KEY}")
print(f"Shape embeddings: {embeddings.shape}")
print(f"Shape npz_chunk_ids: {npz_chunk_ids.shape}")

# ----------------------------
# 6) Validar consistencia parquet vs embeddings (tamaño)
# ----------------------------
n_chunks = len(df_chunks)
n_emb = embeddings.shape[0]

assert n_chunks == n_emb, (
    f"Inconsistencia: parquet tiene {n_chunks} filas, "
    f"pero '{EMB_KEY}' tiene {n_emb} vectores."
)
print("Consistencia de tamaño confirmada (parquet vs embeddings).")

# ----------------------------
# 7) Validar alineación de chunk_id (orden)
# ----------------------------
# Nota: esto valida que el orden de los embeddings coincide con el orden del parquet.
# Si falla, tendremos que hacer alineación por merge con un índice.
assert np.array_equal(df_chunks[ID_COL].values, npz_chunk_ids), (
    "Los chunk_id del parquet NO están alineados (en el mismo orden) con los chunk_id del NPZ.\n"
    "Solución: reordenar embeddings por chunk_id antes de usar cosine similarity."
)
print("Alineación confirmada: chunk_id parquet == chunk_id npz (mismo orden).")

# ----------------------------
# 8) Chequeo de normalización (si aplica)
# ----------------------------
sample_idx = np.random.choice(n_emb, size=200, replace=False)
norms = np.linalg.norm(embeddings[sample_idx], axis=1)

print("\nChequeo de normalización (muestra de 200 vectores):")
print(f"Norma L2 promedio: {norms.mean():.6f}")
print(f"Norma L2 min/max: {norms.min():.6f} / {norms.max():.6f}")

# ----------------------------
# 9) Validar que TEXT_COL es string y no está vacío
# ----------------------------
assert df_chunks[TEXT_COL].dtype == object, f"{TEXT_COL} no parece ser texto (dtype={df_chunks[TEXT_COL].dtype})"

empty_text = (df_chunks[TEXT_COL].str.strip() == "").sum()
assert empty_text == 0, f"Se encontraron {empty_text} chunks con texto vacío en {TEXT_COL}."

print(f"\n{TEXT_COL} validado como texto no vacío.")

# ----------------------------
# 10) Resumen final de artefactos
# ----------------------------

# Cargar manifest (.json) - si aún no está en memoria
t0 = time.time()
with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    manifest = json.load(f)
t_manifest = time.time() - t0

print("\nManifest cargado.")
print(f"Tiempo de carga manifest: {t_manifest:.3f} s")
print(json.dumps(manifest, indent=2, ensure_ascii=False))

print("\n==================== RESUMEN ARTEFACTOS ====================")
print(f"df_chunks:    {df_chunks.shape[0]} filas, {df_chunks.shape[1]} columnas")
print(f"embeddings:   key='{EMB_KEY}', shape={embeddings.shape}")
print(f"model_name:   {manifest.get('model', 'N/A')}")
print(f"normalized:   {manifest.get('normalized', 'N/A')}")
print(f"created_at:   {manifest.get('created_at', 'N/A')}")
print("============================================================")

Columnas clave definidas:
ID_COL   = chunk_id
REF_COL  = chunk_ref
TEXT_COL = chunk_text_rv1909

Parquet cargado.
Tiempo de carga parquet: 2.054 s
Shape df_chunks: (10058, 29)


,chunk_id,book,book_es,chapter,verse_start,verse_end,chunk_ref,chunk_text_rv1909,chunk_text_vbl,testament_AT,...,canon_group_profetas_mayores,canon_group_profetas_menores,canon_group_coarse_AT_historia,canon_group_coarse_AT_ley,canon_group_coarse_AT_poesia_sabiduria,canon_group_coarse_AT_profetas,canon_group_coarse_NT_apocaliptico,canon_group_coarse_NT_cartas,canon_group_coarse_NT_evangelios,canon_group_coarse_NT_historia
0,chk_000000,1CH,1 Crónicas,1,1,4,1CH 1:1-4,"ADAM, Seth, Enos, Cainán, Mahalaleel, Jared, E...","Adán, Set, Enós, Quenán, Malalel, Jared, Enoc,...",True,...,False,False,True,False,False,False,False,False,False,False
1,chk_000001,1CH,1 Crónicas,1,5,8,1CH 1:5-8,"Los hijos de Japhet: Gomer, Magog, Dadai, Javá...","Los hijos de Jafet: Gómer, Magog, Madai, Javan...",True,...,False,False,True,False,False,False,False,False,False,False
2,chk_000002,1CH,1 Crónicas,1,9,12,1CH 1:9-12,"Los hijos de Chûs: Seba, Havila, Sabtha, Raema...","Los hijos de Cus: Seba, Javilá, Sabta, Ragama ...",True,...,False,False,True,False,False,False,False,False,False,False



Columnas requeridas presentes: ['chunk_id', 'chunk_ref', 'chunk_text_rv1909']
No hay valores nulos en columnas críticas.
chunk_id es único (n=10058).

NPZ cargado.
Tiempo de carga embeddings: 0.965 s
Keys disponibles en npz: ['chunk_id', 'emb_rv1909', 'emb_vbl']

Embedding seleccionado: emb_rv1909
Shape embeddings: (10058, 384)
Shape npz_chunk_ids: (10058,)
Consistencia de tamaño confirmada (parquet vs embeddings).
Alineación confirmada: chunk_id parquet == chunk_id npz (mismo orden).

Chequeo de normalización (muestra de 200 vectores):
Norma L2 promedio: 1.000000
Norma L2 min/max: 1.000000 / 1.000000

chunk_text_rv1909 validado como texto no vacío.

Manifest cargado.
Tiempo de carga manifest: 1.643 s
{
  "model": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
  "normalized": true,
  "n_chunks": 10058,
  "created_at": "20260207_084138"
}

==================== RESUMEN ARTEFACTOS ====================
df_chunks:    10058 filas, 29 columnas
embeddings:   key='emb_rv1909', 

### Interpretación (Parte 2)

- El corpus tabular se cargó correctamente desde el `.parquet`, confirmando el tamaño real del dataset de chunks y la presencia de las columnas críticas:
  - `chunk_id` (único)
  - `chunk_ref`
  - `chunk_text_rv1909` (columna de texto principal)

- Los embeddings se cargaron correctamente desde el `.npz`, confirmando la existencia de la key `"embeddings"` y su dimensión `(n_chunks, d)`.

- El `manifest` se cargó correctamente y confirma:
  - El modelo utilizado para embeddings (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`)
  - Si los embeddings fueron normalizados (`normalized=True`), lo cual permite aplicar cosine similarity directamente sin pre-normalización adicional.

- Se verificó consistencia 1:1 entre el parquet y el arreglo de embeddings:
  - El número de filas del corpus coincide con el número de vectores.

- Se confirmó que los embeddings están separados por traducción (RV1909 y VBL), por lo que el pipeline selecciona el embedding correspondiente al TEXT_COL.

- Se confirmó que el archivo `.npz` contiene embeddings separados por traducción (`emb_rv1909` y `emb_vbl`). En este avance, dado que `TEXT_COL = chunk_text_rv1909`, se utiliza explícitamente `emb_rv1909`.

Con estas validaciones completadas, el entorno queda listo para construir los retrievers (BM25, Dense, Hybrid) y definir formalmente los 6 modelos alternativos del Avance 4.

---

> *"Llámame y te responderé; y te mostraré cosas grandes y ocultas que tú no conoces."*
>
> — Jeremías 33:3

---

# **PARTE 3 – Creación del Eval Set v1 (conjunto de evaluación)**

En esta sección se construye un conjunto de evaluación **pequeño pero representativo** para comparar modelos alternativos de LUMINA.

Principios de diseño del eval set:

- **No trivial**: las consultas deben requerir contexto bíblico real, evitando preguntas “de memoria”.
- **Diversidad**: se cubren distintos tipos de necesidades pastorales y teológicas.
- **Reproducible**: se exporta como JSON y CSV para poder reutilizarse en otros avances.
- **Medible**: cada query incluye metadatos (categoría y “pistas” esperadas) para facilitar métricas IR.

Estructura de cada ejemplo (registro):

- `qid`: identificador único
- `category`: categoría temática
- `query`: consulta del usuario
- `expected_refs`: lista opcional de referencias bíblicas esperadas (para métricas IR tipo Recall@k / MRR)
- `notes`: intención de evaluación (qué debe demostrar el modelo)

In [4]:
# ==========================================
# Parte 3 - Construcción del Eval Set v1
# ==========================================

EVAL_DIR = BASE_PATH / "eval_set"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

EVAL_JSON_PATH = EVAL_DIR / "lumina_eval_set_v1.json"
EVAL_CSV_PATH  = EVAL_DIR / "lumina_eval_set_v1.csv"

eval_items = [
    # =========================================================
    # 1) Emocional / Consuelo (5)
    # =========================================================
    {
        "qid": "E01",
        "category": "emocional_consuele",
        "query": "Perdí a un familiar y siento que Dios me abandonó. ¿Qué dice la Biblia para alguien en duelo y sin fuerzas?",
        "expected_refs": ["PSA 23", "MAT 5:4", "JHN 11", "ROM 8:38-39"],
        "notes": "Consuelo pastoral, validación emocional, grounding fuerte con citas pertinentes (duelo, esperanza)."
    },
    {
        "qid": "E02",
        "category": "emocional_consuele",
        "query": "Tengo ansiedad por el futuro y no puedo dormir. ¿Cómo puedo confiar en Dios cuando mi mente no se calla?",
        "expected_refs": ["PHP 4:6-7", "MAT 6:25-34", "PSA 94"],
        "notes": "Debe orientar a oración, confianza y práctica espiritual sin caer en 'solo échale ganas'."
    },
    {
        "qid": "E03",
        "category": "emocional_consuele",
        "query": "Siento culpa por errores del pasado y creo que Dios no me puede perdonar. ¿Qué enseña la Biblia sobre el perdón?",
        "expected_refs": ["1JHN 1:9", "PSA 51", "LUK 15"],
        "notes": "Debe explicar gracia, arrepentimiento y restauración. Citas directas sobre perdón."
    },
    {
        "qid": "E04",
        "category": "emocional_consuele",
        "query": "Estoy agotado y sin motivación espiritual. ¿Qué hago cuando ya no siento nada al orar?",
        "expected_refs": ["PSA 42", "ISA 40:29-31", "MAT 11:28-30"],
        "notes": "Debe normalizar sequedad espiritual, dar pasos concretos y sostener con texto bíblico."
    },
    {
        "qid": "E05",
        "category": "emocional_consuele",
        "query": "Tengo miedo constante a la muerte. ¿Cómo habla la Biblia del temor y de la vida eterna?",
        "expected_refs": ["HEB 2:14-15", "JHN 14:1-3", "1COR 15"],
        "notes": "Debe hablar de esperanza cristiana y vencer temor con grounding (resurrección, promesa)."
    },

    # =========================================================
    # 2) Doctrinal / Teología (5)
    # =========================================================
    {
        "qid": "D01",
        "category": "doctrinal_teologia",
        "query": "¿Qué significa realmente 'ser salvo por gracia' y no por obras?",
        "expected_refs": ["EPH 2:8-10", "ROM 3", "TIT 3:5"],
        "notes": "Explicar doctrina con claridad y límites: obras como fruto, no causa."
    },
    {
        "qid": "D02",
        "category": "doctrinal_teologia",
        "query": "¿Qué enseña la Biblia sobre el Espíritu Santo y su obra en un creyente?",
        "expected_refs": ["JHN 14:16-17", "ACT 1:8", "GAL 5:22-23"],
        "notes": "Debe citar pasajes sobre Consolador, poder, fruto del Espíritu."
    },
    {
        "qid": "D03",
        "category": "doctrinal_teologia",
        "query": "¿Por qué Jesús tenía que morir? Explícalo bíblicamente, no solo con opinión.",
        "expected_refs": ["ISA 53", "ROM 5:8", "1PET 2:24"],
        "notes": "Expiación/sustitución. Grounding fuerte. Tono pastoral."
    },
    {
        "qid": "D04",
        "category": "doctrinal_teologia",
        "query": "¿Qué es el arrepentimiento según la Biblia y cómo se ve en la práctica?",
        "expected_refs": ["ACT 2:38", "2COR 7:10", "LUK 19"],
        "notes": "Debe diferenciar remordimiento vs arrepentimiento, incluir fruto visible."
    },
    {
        "qid": "D05",
        "category": "doctrinal_teologia",
        "query": "¿Cómo define la Biblia el amor? No solo amor romántico; amor en sentido cristiano.",
        "expected_refs": ["1COR 13", "1JHN 4:7-12", "JHN 13:34-35"],
        "notes": "Debe citar definiciones y aplicaciones prácticas."
    },

    # =========================================================
    # 3) Ética / Decisiones (5)
    # =========================================================
    {
        "qid": "ET01",
        "category": "etica_decisiones",
        "query": "Estoy considerando mentir para evitar un problema grande. ¿Qué principios bíblicos aplican aquí?",
        "expected_refs": ["PRO 12:22", "EPH 4:25", "COL 3:9"],
        "notes": "Debe condenar mentira y proponer alternativas (verdad con sabiduría)."
    },
    {
        "qid": "ET02",
        "category": "etica_decisiones",
        "query": "¿Qué dice la Biblia sobre perdonar a alguien que me hizo daño pero no se arrepiente?",
        "expected_refs": ["MAT 18", "ROM 12:17-21", "LUK 23:34"],
        "notes": "Debe equilibrar perdón con límites/justicia, sin habilitar abuso."
    },
    {
        "qid": "ET03",
        "category": "etica_decisiones",
        "query": "Tengo dinero extra, ¿cómo enseña la Biblia que debo administrarlo?",
        "expected_refs": ["PRO 3:9-10", "1TIM 6:17-19", "LUK 16"],
        "notes": "Mayordomía, generosidad, evitar amor al dinero."
    },
    {
        "qid": "ET04",
        "category": "etica_decisiones",
        "query": "Estoy enojado todo el tiempo y exploto con mi familia. ¿Qué dice la Biblia sobre la ira?",
        "expected_refs": ["EPH 4:26-27", "JAS 1:19-20", "PRO 15:1"],
        "notes": "Debe dar guía práctica y citar textos sobre controlar ira."
    },
    {
        "qid": "ET05",
        "category": "etica_decisiones",
        "query": "Trabajo en algo que no es ilegal, pero siento que no es correcto. ¿Cómo discierno bíblicamente?",
        "expected_refs": ["ROM 14", "1COR 10:23", "PHP 1:9-10"],
        "notes": "Discernimiento, conciencia, 'todo me es lícito pero no todo conviene'."
    },

    # =========================================================
    # 4) Narrativa / Contexto bíblico (5)
    # =========================================================
    {
        "qid": "N01",
        "category": "narrativa_contexto",
        "query": "Resume qué pasó en el éxodo y qué significado tiene para la fe hoy.",
        "expected_refs": ["EXO 12-14", "1COR 10:1-4"],
        "notes": "Debe narrar y conectar tipología (liberación) sin inventar detalles."
    },
    {
        "qid": "N02",
        "category": "narrativa_contexto",
        "query": "¿Por qué David fue considerado 'un hombre conforme al corazón de Dios' si cometió errores graves?",
        "expected_refs": ["1SA 13:14", "PSA 51", "2SA 11-12"],
        "notes": "Debe explicar arrepentimiento y misericordia, con contexto narrativo."
    },
    {
        "qid": "N03",
        "category": "narrativa_contexto",
        "query": "¿Qué enseña la historia de Job sobre el sufrimiento y la soberanía de Dios?",
        "expected_refs": ["JOB 1-2", "JOB 38-42", "JAS 5:11"],
        "notes": "Debe evitar simplismos, incluir límites humanos y carácter de Dios."
    },
    {
        "qid": "N04",
        "category": "narrativa_contexto",
        "query": "¿Qué es el 'Sermón del Monte' y cuáles son sus ideas centrales?",
        "expected_refs": ["MAT 5-7"],
        "notes": "Debe citar y sintetizar correctamente (bienaventuranzas, ética del reino)."
    },
    {
        "qid": "N05",
        "category": "narrativa_contexto",
        "query": "¿Qué significa que Pablo hable del 'fruto del Espíritu'? ¿En qué contexto lo dijo?",
        "expected_refs": ["GAL 5:16-26"],
        "notes": "Debe mencionar conflicto carne vs Espíritu y listar fruto correctamente."
    },

    # =========================================================
    # 5) Sufrimiento / Crisis (5)
    # =========================================================
    {
        "qid": "S01",
        "category": "sufrimiento_crisis",
        "query": "Estoy pasando una enfermedad larga y me siento inútil. ¿Cómo ver el sufrimiento a la luz de la Biblia?",
        "expected_refs": ["ROM 5:3-5", "2COR 12:9-10", "PSA 73"],
        "notes": "Debe dar esperanza sin prometer sanidad inmediata. Grounding bíblico."
    },
    {
        "qid": "S02",
        "category": "sufrimiento_crisis",
        "query": "¿Por qué Dios permite injusticias si es bueno? Dame una respuesta bíblica honesta.",
        "expected_refs": ["HAB 1-3", "ROM 9", "REV 21"],
        "notes": "Debe reconocer tensión, citar lamento y esperanza escatológica."
    },
    {
        "qid": "S03",
        "category": "sufrimiento_crisis",
        "query": "Me siento tentado a rendirme con mi vida. ¿Cómo responde la Biblia ante la desesperación?",
        "expected_refs": ["PSA 34:18", "1KGS 19", "MAT 11:28-30"],
        "notes": "Tono pastoral cuidadoso, esperanza, buscar ayuda; grounding fuerte."
    },
    {
        "qid": "S04",
        "category": "sufrimiento_crisis",
        "query": "Estoy quebrado por culpa y vergüenza. ¿Qué dice la Biblia sobre restauración después de caer?",
        "expected_refs": ["PSA 51", "JHN 21", "ROM 8:1"],
        "notes": "Restauración, gracia, arrepentimiento. Citas claras."
    },
    {
        "qid": "S05",
        "category": "sufrimiento_crisis",
        "query": "¿Cómo puedo mantener esperanza cuando mis oraciones parecen no tener respuesta?",
        "expected_refs": ["PSA 13", "LUK 18:1-8", "JAS 1:2-4"],
        "notes": "Perseverancia, lamento bíblico, fe práctica."
    },

    # =========================================================
    # 6) Discernimiento / Tentación (5)
    # =========================================================
    {
        "qid": "DI01",
        "category": "discernimiento_tentacion",
        "query": "Estoy siendo tentado constantemente con algo que sé que me hace daño. ¿Qué pasos bíblicos puedo seguir?",
        "expected_refs": ["1COR 10:13", "MAT 26:41", "PSA 119:9-11"],
        "notes": "Debe dar pasos prácticos (huir, vigilar, palabra) con citas."
    },
    {
        "qid": "DI02",
        "category": "discernimiento_tentacion",
        "query": "¿Cómo discierno si una decisión viene de Dios o solo de mis deseos?",
        "expected_refs": ["PRO 3:5-6", "JAS 1:5", "1JHN 4:1"],
        "notes": "Sabiduría, consejo, prueba de espíritus, humildad."
    },
    {
        "qid": "DI03",
        "category": "discernimiento_tentacion",
        "query": "Siento envidia cuando otros prosperan. ¿Qué enseña la Biblia sobre la envidia y el contentamiento?",
        "expected_refs": ["PSA 73", "PHI 4:11-13", "PRO 14:30"],
        "notes": "Debe confrontar envidia con contentamiento y perspectiva eterna."
    },
    {
        "qid": "DI04",
        "category": "discernimiento_tentacion",
        "query": "¿Cómo debo responder cuando alguien me provoca y quiero vengarme?",
        "expected_refs": ["ROM 12:19-21", "MAT 5:38-48", "1PET 2:23"],
        "notes": "No venganza, amor al enemigo, imitar a Cristo."
    },
    {
        "qid": "DI05",
        "category": "discernimiento_tentacion",
        "query": "Quiero crecer espiritualmente, pero siempre vuelvo a los mismos hábitos. ¿Qué enseña la Biblia sobre santificación?",
        "expected_refs": ["ROM 6", "GAL 5", "1TH 4:3"],
        "notes": "Proceso, dependencia del Espíritu, disciplina espiritual."
    },
]

# Validación rápida
assert len(eval_items) == 30, f"Se esperaban 30 items, pero hay {len(eval_items)}."

# Convertir a DataFrame
df_eval = pd.DataFrame(eval_items)

# Export JSON y CSV
df_eval.to_json(EVAL_JSON_PATH, orient="records", force_ascii=False, indent=2)
df_eval.to_csv(EVAL_CSV_PATH, index=False, encoding="utf-8")

print("Eval set v1 creado y exportado.")
print("JSON:", EVAL_JSON_PATH)
print("CSV: ", EVAL_CSV_PATH)

display(df_eval.head(10))

Eval set v1 creado y exportado.
JSON: /content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/Avance 4 - Modelos alternativos/Avance 4 - Steven/eval_set/lumina_eval_set_v1.json
CSV:  /content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/Avance 4 - Modelos alternativos/Avance 4 - Steven/eval_set/lumina_eval_set_v1.csv


,qid,category,query,expected_refs,notes
0,E01,emocional_consuele,Perdí a un familiar y siento que Dios me aband...,"[PSA 23, MAT 5:4, JHN 11, ROM 8:38-39]","Consuelo pastoral, validación emocional, groun..."
1,E02,emocional_consuele,Tengo ansiedad por el futuro y no puedo dormir...,"[PHP 4:6-7, MAT 6:25-34, PSA 94]","Debe orientar a oración, confianza y práctica ..."
2,E03,emocional_consuele,Siento culpa por errores del pasado y creo que...,"[1JHN 1:9, PSA 51, LUK 15]","Debe explicar gracia, arrepentimiento y restau..."
3,E04,emocional_consuele,Estoy agotado y sin motivación espiritual. ¿Qu...,"[PSA 42, ISA 40:29-31, MAT 11:28-30]","Debe normalizar sequedad espiritual, dar pasos..."
4,E05,emocional_consuele,Tengo miedo constante a la muerte. ¿Cómo habla...,"[HEB 2:14-15, JHN 14:1-3, 1COR 15]",Debe hablar de esperanza cristiana y vencer te...
5,D01,doctrinal_teologia,¿Qué significa realmente 'ser salvo por gracia...,"[EPH 2:8-10, ROM 3, TIT 3:5]",Explicar doctrina con claridad y límites: obra...
6,D02,doctrinal_teologia,¿Qué enseña la Biblia sobre el Espíritu Santo ...,"[JHN 14:16-17, ACT 1:8, GAL 5:22-23]","Debe citar pasajes sobre Consolador, poder, fr..."
7,D03,doctrinal_teologia,¿Por qué Jesús tenía que morir? Explícalo bíbl...,"[ISA 53, ROM 5:8, 1PET 2:24]",Expiación/sustitución. Grounding fuerte. Tono ...
8,D04,doctrinal_teologia,¿Qué es el arrepentimiento según la Biblia y c...,"[ACT 2:38, 2COR 7:10, LUK 19]",Debe diferenciar remordimiento vs arrepentimie...
9,D05,doctrinal_teologia,¿Cómo define la Biblia el amor? No solo amor r...,"[1COR 13, 1JHN 4:7-12, JHN 13:34-35]",Debe citar definiciones y aplicaciones prácticas.


### Interpretación (Parte 3)

- Se construyó un conjunto de evaluación **v1** con 30 consultas, organizado en 6 categorías (5 por categoría).
- Cada ejemplo incluye:
  - Identificador (`qid`)
  - Categoría (`category`)
  - Consulta (`query`)
  - Referencias esperadas (`expected_refs`) para métricas IR (Recall@k / MRR)
  - Notas de intención (`notes`) para guiar el análisis cualitativo

- El conjunto fue exportado en dos formatos:
  - `lumina_eval_set_v1.json`
  - `lumina_eval_set_v1.csv`

Este eval set servirá como base reproducible para comparar el desempeño de los 6 modelos alternativos y posteriormente ajustar los dos mejores.

---

> *"Además de ser sabio, el Maestro impartió conocimientos a la gente. Escuchó, investigó y ordenó con cuidado muchos proverbios."*
>
> — Eclesiastés 12:9

---

# **PARTE 4 – Definición formal de los 6 modelos alternativos**

En LUMINA, un **modelo** se define como una configuración completa del pipeline:

**Retriever + Representación + Prompt + LLM + Hiperparámetros + Post-procesamiento (si aplica)**

Esto es consistente con el enfoque de sistemas RAG, donde el desempeño depende de múltiples componentes.

En este avance se construirán al menos **6 modelos individuales** (no ensambles), variando explícitamente:

- **Algoritmo de recuperación**: LLM-only, BM25, Dense, Hybrid
- **Representación semántica**: SentenceTransformers (local) vs OpenAI embeddings (API)
- **Arquitectura del pipeline**: con y sin reranking (cross-encoder)
- **Parámetros**: k, alpha, temperature, etc.

La comparación se realizará sobre el mismo conjunto de evaluación (Eval Set v1) para asegurar equidad experimental.

In [5]:
# ==========================================
# Parte 4 - Definición de modelos (configs)
# ==========================================

# ----------------------------
# 1) Prompt base (pastoral + grounding)
# ----------------------------
SYSTEM_PROMPT_BASE = """
Eres LUMINA, un asistente pastoral cristiano basado exclusivamente en la Biblia.
Tu tarea es responder de forma compasiva, clara y bíblicamente fiel.

REGLAS OBLIGATORIAS:
1) Basa la respuesta en los pasajes proporcionados (si hay).
2) NO inventes versículos ni referencias.
3) SI no hay información suficiente en los pasajes, dilo explícitamente.
4) Usa un tono pastoral y práctico.
5) Incluye al final una sección: "Citas bíblicas" con referencias exactas.
"""

USER_PROMPT_RAG_TEMPLATE = """
Pregunta del usuario:
{query}

Pasajes bíblicos recuperados:
{context}

Instrucciones:
- Responde en español.
- Da una respuesta pastoral y práctica.
- Sustenta con los pasajes proporcionados.
- No inventes referencias.

Formato de salida:
1) Respuesta pastoral (1-3 párrafos)
2) Aplicación práctica (bullets)
3) Citas bíblicas (lista de referencias exactas)
"""

USER_PROMPT_LLM_ONLY_TEMPLATE = """
Pregunta del usuario:
{query}

Instrucciones:
- Responde en español.
- Da una respuesta pastoral y práctica.
- No inventes citas específicas si no estás seguro.
- Si no puedes citar con precisión, explica el principio bíblico sin citar versículos exactos.

Formato de salida:
1) Respuesta pastoral (1-3 párrafos)
2) Aplicación práctica (bullets)
3) (Opcional) Referencias generales (sin inventar)
"""

# ----------------------------
# 2) Estructura estándar de "modelo"
# ----------------------------
# Nota: Estas configs no ejecutan aún; solo definen variantes reproducibles.

MODELS = {
    # =========================================================
    # M1 — LLM-only
    # =========================================================
    "M1_LLM_ONLY": {
        "model_id": "M1_LLM_ONLY",
        "description": "LLM-only (sin retrieval). Baseline generativo puro.",
        "pipeline": {
            "retriever": "none",
            "reranker": "none",
            "embedding_source": "none"
        },
        "params": {
            "temperature": 0.2,
            "top_p": 1.0,
            "max_tokens": 500
        },
        "prompt": {
            "system": SYSTEM_PROMPT_BASE,
            "user_template": USER_PROMPT_LLM_ONLY_TEMPLATE
        }
    },

    # =========================================================
    # M2 — BM25-RAG
    # =========================================================
    "M2_BM25_RAG": {
        "model_id": "M2_BM25_RAG",
        "description": "BM25-RAG (retrieval léxico clásico) + generación con LLM.",
        "pipeline": {
            "retriever": "bm25",
            "reranker": "none",
            "embedding_source": "text"
        },
        "params": {
            "k": 3,
            "bm25_k1": 1.5,
            "bm25_b": 0.75,
            "temperature": 0.2,
            "top_p": 1.0,
            "max_tokens": 500
        },
        "prompt": {
            "system": SYSTEM_PROMPT_BASE,
            "user_template": USER_PROMPT_RAG_TEMPLATE
        }
    },

    # =========================================================
    # M3 — Dense-RAG (SentenceTransformers local)
    # =========================================================
    "M3_DENSE_ST_RAG": {
        "model_id": "M3_DENSE_ST_RAG",
        "description": "Dense-RAG usando embeddings locales SentenceTransformers (cosine).",
        "pipeline": {
            "retriever": "dense_st",
            "reranker": "none",
            "embedding_source": "local_npz",
            "embedding_key": EMB_KEY  # emb_rv1909
        },
        "params": {
            "k": 3,
            "temperature": 0.2,
            "top_p": 1.0,
            "max_tokens": 500
        },
        "prompt": {
            "system": SYSTEM_PROMPT_BASE,
            "user_template": USER_PROMPT_RAG_TEMPLATE
        }
    },

    # =========================================================
    # M4 — Hybrid-RAG (BM25 + Dense)
    # =========================================================
    "M4_HYBRID_RAG": {
        "model_id": "M4_HYBRID_RAG",
        "description": "Hybrid-RAG combinando BM25 + Dense con alpha fijo.",
        "pipeline": {
            "retriever": "hybrid",
            "reranker": "none",
            "embedding_source": "local_npz",
            "embedding_key": EMB_KEY
        },
        "params": {
            "k": 3,
            "alpha": 0.2,           # peso para dense (ej. score = alpha*dense + (1-alpha)*bm25)
            "bm25_k1": 1.5,
            "bm25_b": 0.75,
            "temperature": 0.2,
            "top_p": 1.0,
            "max_tokens": 500
        },
        "prompt": {
            "system": SYSTEM_PROMPT_BASE,
            "user_template": USER_PROMPT_RAG_TEMPLATE
        }
    },

    # =========================================================
    # M5 — Dense-RAG con embeddings OpenAI (text-embedding-3-small)
    # =========================================================
    # Nota: este modelo requiere generar embeddings de chunks y queries vía API.
    # Lo definimos desde ya como config reproducible, aunque su ejecución se implementará después.
    "M5_DENSE_OAI_RAG": {
        "model_id": "M5_DENSE_OAI_RAG",
        "description": "Dense-RAG usando embeddings OpenAI (text-embedding-3-small).",
        "pipeline": {
            "retriever": "dense_openai",
            "reranker": "none",
            "embedding_source": "openai_api",
            "embedding_model": "text-embedding-3-small"
        },
        "params": {
            "k": 3,
            "temperature": 0.2,
            "top_p": 1.0,
            "max_tokens": 500
        },
        "prompt": {
            "system": SYSTEM_PROMPT_BASE,
            "user_template": USER_PROMPT_RAG_TEMPLATE
        }
    },

    # =========================================================
    # M6 — Dense-RAG + Reranker (Cross-Encoder)
    # =========================================================
    # Nota: el reranker reordena un top_n inicial (ej. 10) y luego se queda con top_k final (ej. 3).
    "M6_DENSE_ST_RERANK": {
        "model_id": "M6_DENSE_ST_RERANK",
        "description": "Dense-RAG (ST) + reranking con Cross-Encoder sobre top_n candidatos.",
        "pipeline": {
            "retriever": "dense_st",
            "reranker": "cross_encoder",
            "embedding_source": "local_npz",
            "embedding_key": EMB_KEY,
            "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2"
        },
        "params": {
            "top_n": 10,    # candidatos iniciales del retriever dense
            "k": 3,         # top-k final después de rerank
            "temperature": 0.2,
            "top_p": 1.0,
            "max_tokens": 500
        },
        "prompt": {
            "system": SYSTEM_PROMPT_BASE,
            "user_template": USER_PROMPT_RAG_TEMPLATE
        }
    }
}

print(f"Se definieron {len(MODELS)} modelos alternativos.")
print("IDs:", list(MODELS.keys()))

# Vista compacta
pd.DataFrame([
    {
        "model_id": cfg["model_id"],
        "retriever": cfg["pipeline"]["retriever"],
        "reranker": cfg["pipeline"]["reranker"],
        "embedding_source": cfg["pipeline"]["embedding_source"],
        "k": cfg["params"].get("k", None),
        "temperature": cfg["params"].get("temperature", None)
    }
    for cfg in MODELS.values()
])

Se definieron 6 modelos alternativos.
IDs: ['M1_LLM_ONLY', 'M2_BM25_RAG', 'M3_DENSE_ST_RAG', 'M4_HYBRID_RAG', 'M5_DENSE_OAI_RAG', 'M6_DENSE_ST_RERANK']


,model_id,retriever,reranker,embedding_source,k,temperature
0,M1_LLM_ONLY,none,none,none,NaN,0.2
1,M2_BM25_RAG,bm25,none,text,3.0,0.2
2,M3_DENSE_ST_RAG,dense_st,none,local_npz,3.0,0.2
3,M4_HYBRID_RAG,hybrid,none,local_npz,3.0,0.2
4,M5_DENSE_OAI_RAG,dense_openai,none,openai_api,3.0,0.2
5,M6_DENSE_ST_RERANK,dense_st,cross_encoder,local_npz,3.0,0.2


### Interpretación

- Se definieron formalmente 6 modelos individuales, cada uno como una configuración completa del pipeline LUMINA.
- La variación entre modelos incluye:
  - Ausencia/presencia de retrieval (LLM-only vs RAG).
  - Recuperación léxica (BM25), semántica (Dense) e híbrida (Hybrid).
  - Diferente fuente de embeddings (local SentenceTransformers vs OpenAI embeddings).
  - Diferente arquitectura interna del pipeline (con reranking mediante Cross-Encoder en M6).

Estas configuraciones servirán como base reproducible para:
1) Implementar los retrievers y el reranker.
2) Ejecutar evaluación batch sobre el Eval Set v1.
3) Construir la tabla comparativa final.
4) Seleccionar Top-2 y realizar ajuste fino en el Avance 4.

---

> *"Pero todo lo que se pone bajo la luz se hace visible, y lo que se hace visible se convierte en luz."*
>
> — Efesios 5:13-14

---

# **PARTE 5 – Implementación de retrievers (BM25, Dense, Hybrid) y reranker (Cross-Encoder)**

En esta sección se implementan los componentes de recuperación necesarios para ejecutar los modelos definidos en la Parte 4:

- **BM25 Retriever**: recuperación léxica basada en coincidencia de términos.
- **Dense Retriever**: recuperación semántica mediante cosine similarity sobre embeddings precomputados.
- **Hybrid Retriever**: combinación lineal de puntajes BM25 y Dense con un parámetro α.
- **Reranker (Cross-Encoder)**: re-ordena los candidatos top-n usando un modelo cross-encoder para seleccionar el top-k final.

Se construyen funciones y estructuras reutilizables para que:
- Cada modelo pueda invocar el retriever correspondiente de forma uniforme.
- Se registren tiempos (latencia) y resultados recuperados.
- El proceso sea reproducible y fácil de ajustar en el tuning (Parte 9).

In [6]:
# ==========================================
# 5.1 - Preparación de datos para retrieval (textos, referencias, tokenización)
# ==========================================

# ----------------------------
# 1) Vectores/arrays base para acceso rápido
# ----------------------------
chunk_ids = df_chunks[ID_COL].values
chunk_refs = df_chunks[REF_COL].values
chunk_texts = df_chunks[TEXT_COL].values

# ----------------------------
# 2) Tokenización simple para BM25
#    - lower()
#    - split básico
#    (suficiente para baseline; en tuning podemos mejorar)
# ----------------------------
def simple_tokenize(text: str):
    return str(text).lower().split()

tokenized_corpus = [simple_tokenize(t) for t in chunk_texts]

print("Preparación lista:")
print(f"- tokenized_corpus: {len(tokenized_corpus)} documentos")
print(f"- embeddings shape: {embeddings.shape}")

Preparación lista:
- tokenized_corpus: 10058 documentos
- embeddings shape: (10058, 384)


In [7]:
# ==========================================
# 5.2 - BM25 Retriever (build + query)
# ==========================================

def build_bm25_index(tokenized_corpus, k1=1.5, b=0.75):
    """
    Construye un índice BM25Okapi con hiperparámetros configurables.
    """
    bm25 = BM25Okapi(tokenized_corpus, k1=k1, b=b)
    return bm25

def bm25_retrieve(bm25, query: str, k: int = 3):
    """
    Retorna top-k documentos usando BM25.
    Output: list of tuples (idx, score)
    """
    q_tokens = simple_tokenize(query)
    scores = bm25.get_scores(q_tokens)  # array size n_docs
    top_idx = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_idx]

# Construcción inicial con parámetros default (los de M2/M4)
t0 = time.time()
bm25_index = build_bm25_index(tokenized_corpus, k1=1.5, b=0.75)
t_bm25_build = time.time() - t0

print(f"BM25 index construido. Tiempo build: {t_bm25_build:.3f} s")

BM25 index construido. Tiempo build: 0.690 s


In [8]:
# ==========================================
# 5.3 – Dense Retriever (cosine similarity sobre embeddings normalizados)
# ==========================================

def dense_retrieve(query_emb: np.ndarray, doc_embs: np.ndarray, k: int = 3):
    """
    Recuperación Dense por cosine similarity.

    Asume embeddings normalizados => cosine = dot product.
    Output: list of tuples (idx, score)
    """
    # query_emb shape: (d,) o (1,d)
    if query_emb.ndim == 1:
        query_emb = query_emb.reshape(1, -1)

    # Cosine sim: (1,d) dot (n,d)^T => (1,n)
    sims = np.dot(query_emb, doc_embs.T).flatten()
    top_idx = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i])) for i in top_idx]

print("Dense retrieval listo (función definida).")

Dense retrieval listo (función definida).


In [9]:
# ==========================================
# 5.4 – Hybrid Retriever (α*dense + (1-α)*bm25)
# ==========================================

def minmax_scale(scores: np.ndarray):
    """
    Escala scores a [0,1] para combinar BM25 y Dense.
    Evita que una escala domine a la otra.
    """
    s_min, s_max = scores.min(), scores.max()
    if s_max - s_min < 1e-12:
        return np.zeros_like(scores)
    return (scores - s_min) / (s_max - s_min)

def hybrid_retrieve(bm25, query: str, query_emb: np.ndarray, doc_embs: np.ndarray,
                    k: int = 3, alpha: float = 0.2):
    """
    Combina BM25 y Dense.
    score_final = alpha * dense_scaled + (1 - alpha) * bm25_scaled
    """
    # BM25 scores
    q_tokens = simple_tokenize(query)
    bm25_scores = np.array(bm25.get_scores(q_tokens), dtype=np.float32)

    # Dense sims
    if query_emb.ndim == 1:
        query_emb = query_emb.reshape(1, -1)
    dense_scores = np.dot(query_emb, doc_embs.T).flatten().astype(np.float32)

    # Normalizar escalas
    bm25_scaled = minmax_scale(bm25_scores)
    dense_scaled = minmax_scale(dense_scores)

    final_scores = alpha * dense_scaled + (1 - alpha) * bm25_scaled
    top_idx = np.argsort(final_scores)[::-1][:k]

    return [(int(i), float(final_scores[i])) for i in top_idx]

print("Hybrid retrieval listo (función definida).")

Hybrid retrieval listo (función definida).


In [10]:
# ==========================================
# 5.5 – Reranker (Cross-Encoder) para M6
# ==========================================

from sentence_transformers import CrossEncoder

# Cargar modelo de reranking una sola vez
t0 = time.time()
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
t_ce_load = time.time() - t0

print(f"Cross-Encoder cargado. Tiempo load: {t_ce_load:.3f} s")

def rerank_cross_encoder(query: str, candidate_indices, top_k: int = 3):
    """
    Reordena candidatos usando Cross-Encoder.
    candidate_indices: lista de índices (int) de df_chunks.
    Output: list of tuples (idx, rerank_score) top_k
    """
    pairs = [(query, chunk_texts[i]) for i in candidate_indices]
    scores = cross_encoder.predict(pairs)  # array
    scores = np.array(scores, dtype=np.float32)

    order = np.argsort(scores)[::-1][:top_k]
    reranked = [(int(candidate_indices[i]), float(scores[i])) for i in order]
    return reranked

print("Reranker listo (función definida).")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-Encoder cargado. Tiempo load: 5.377 s
Reranker listo (función definida).


### Interpretación (Parte 5)

- Se preparó el corpus para recuperación:
  - Tokenización básica para BM25.
  - Arreglos en memoria para acceso rápido a `chunk_id`, `chunk_ref` y texto.

- Se implementaron tres retrievers:
  - **BM25**: recuperación léxica con índice construible y parámetros (`k1`, `b`).
  - **Dense**: recuperación semántica vía cosine similarity (dot product por normalización).
  - **Hybrid**: combinación de puntajes BM25 y Dense mediante escalado min-max y peso α.

- Se implementó un **reranker Cross-Encoder** (M6), que reordena un top-n inicial de candidatos y selecciona el top-k final.

Con estos componentes, el sistema queda listo para construir un `runner` unificado (Parte 6), capaz de ejecutar cualquier configuración de `MODELS` midiendo tiempos y recolectando resultados.

---

> *"La balanza justa y los pesos exactos son del Señor; él es quien establece todas las pesas de la bolsa."*
>
> — Proverbios 16:11

---

# **PARTE 6 – Runner unificado para ejecución de modelos (con caching y medición de tiempos)**

En esta sección se construye un `runner` unificado para ejecutar cualquier modelo definido en `MODELS`.

El runner estandariza:

1) **Retrieval** (si aplica):
   - BM25
   - Dense (SentenceTransformers local)
   - Hybrid (BM25 + Dense)
   - Dense + Reranking (Cross-Encoder)

2) **Construcción de contexto**:
   - Se concatenan los pasajes recuperados.
   - Se agrega referencia explícita por chunk.
   - Se controla longitud para no exceder límites.

3) **Generación con LLM** (OpenAI API):
   - Se usa un prompt estructurado.
   - Se registran tokens y tiempos.

4) **Caching**:
   - Para ahorrar costo y tiempo, los resultados se guardan en disco (JSON).
   - Si ya existe una ejecución para (modelo, query), se reutiliza.

El output final por (modelo, query) incluye:
- Respuesta generada
- Citas detectadas
- Chunks recuperados (ids + refs + scores)
- Tiempos (retrieval, generación, total)

In [11]:
# ==========================================
# 6.1 - Setup OpenAI + caching + utilidades
# ==========================================

import hashlib
from datetime import datetime

# ---------
# OpenAI API
# ---------

# Tomamos el API Key de OpenAI desde 'secrets' en Colab

from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

if not OPENAI_API_KEY:
    print("No se encontró OPENAI_API_KEY en el entorno.")
else:
    print("OPENAI_API_KEY encontrada en el entorno (no se imprime por seguridad).")

# Importar OpenAI SDK (nuevo)
try:
    from openai import OpenAI
except Exception as e:
    raise ImportError(
        "No se pudo importar 'openai'.\n"
        "Solución típica en Colab:\n"
        "  !pip -q install openai\n"
        "Luego reinicia runtime si es necesario.\n"
        f"Detalle: {e}"
    )

client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None

GEN_MODEL_NAME = "gpt-4o-mini"   # control de costos (como en Avance 3)

# ----------------------------
# Caching a disco
# ----------------------------
CACHE_DIR = BASE_PATH / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

RUNS_DIR = BASE_PATH / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def _stable_hash(text: str) -> str:
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def cache_path(model_id: str, query: str) -> Path:
    h = _stable_hash(query)
    return CACHE_DIR / f"{model_id}__{h}.json"

# ----------------------------
# Construcción de contexto (RAG)
# ----------------------------
def build_context(retrieved, max_chars: int = 5000):
    """
    retrieved: list of dicts con keys: idx, score, chunk_id, chunk_ref, chunk_text
    Retorna:
    - context_str: texto para prompt
    - citations: lista de referencias usadas
    """
    parts = []
    citations = []

    total = 0
    for r in retrieved:
        ref = r["chunk_ref"]
        txt = r["chunk_text"]

        block = f"[{ref} | {r['chunk_id']}]\n{txt}\n"
        if total + len(block) > max_chars:
            break

        parts.append(block)
        citations.append(ref)
        total += len(block)

    context_str = "\n---\n".join(parts)
    return context_str, citations

# ----------------------------
# Llamada LLM (OpenAI)
# ----------------------------
def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2,
             top_p: float = 1.0, max_tokens: int = 500):
    """
    Llama a OpenAI Chat Completions (Responses API style via openai-python).
    Retorna: text, usage(dict o None), elapsed_time
    """
    t0 = time.time()
    resp = client.chat.completions.create(
        model=GEN_MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_prompt.strip()}
        ],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )
    elapsed = time.time() - t0

    text = resp.choices[0].message.content
    usage = None
    try:
        usage = resp.usage.model_dump() if hasattr(resp.usage, "model_dump") else dict(resp.usage)
    except Exception:
        usage = None

    return text, usage, elapsed

print("Setup OpenAI + caching listo.")
print("GEN_MODEL_NAME:", GEN_MODEL_NAME)
print("CACHE_DIR:", CACHE_DIR)

OPENAI_API_KEY encontrada en el entorno (no se imprime por seguridad).
Setup OpenAI + caching listo.
GEN_MODEL_NAME: gpt-4o-mini
CACHE_DIR: /content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/Avance 4 - Modelos alternativos/Avance 4 - Steven/cache


In [12]:
# ==========================================
# 6.2 - Query embeddings para modelos Dense (ST local)
# ==========================================

from sentence_transformers import SentenceTransformer

# Cargar el mismo modelo del manifest para consistencia experimental
ST_MODEL_NAME = manifest["model"]

t0 = time.time()
st_model = SentenceTransformer(ST_MODEL_NAME)
t_st_load = time.time() - t0

print(f"SentenceTransformer cargado: {ST_MODEL_NAME}")
print(f"Tiempo load ST: {t_st_load:.3f} s")

def embed_query_st(query: str) -> np.ndarray:
    """
    Genera embedding normalizado para una query usando ST local.
    """
    emb = st_model.encode([query], normalize_embeddings=True)  # (1,d)
    return emb[0]  # (d,)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer cargado: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Tiempo load ST: 10.767 s


In [13]:
# ==========================================
# 6.3 - Retrieval dispatcher (función unificada de retrieval por modelo)
# ==========================================

def retrieve_for_model(model_cfg: dict, query: str):
    """
    Retorna:
    - retrieved_docs: list[dict] con idx/score/chunk_id/chunk_ref/chunk_text
    - retrieval_time
    """
    retriever = model_cfg["pipeline"]["retriever"]
    params = model_cfg["params"]

    t0 = time.time()

    # ----------------------------
    # No retrieval
    # ----------------------------
    if retriever == "none":
        return [], time.time() - t0

    # ----------------------------
    # BM25
    # ----------------------------
    if retriever == "bm25":
        k = int(params.get("k", 3))
        hits = bm25_retrieve(bm25_index, query, k=k)

        retrieved = []
        for idx, score in hits:
            retrieved.append({
                "idx": idx,
                "score": score,
                "chunk_id": chunk_ids[idx],
                "chunk_ref": chunk_refs[idx],
                "chunk_text": chunk_texts[idx]
            })
        return retrieved, time.time() - t0

    # ----------------------------
    # Dense ST local
    # ----------------------------
    if retriever == "dense_st":
        k = int(params.get("k", 3))
        # Para M6 primero recuperamos top_n y luego rerankeamos
        top_n = int(params.get("top_n", k))

        q_emb = embed_query_st(query)
        hits = dense_retrieve(q_emb, embeddings, k=top_n)

        candidate_indices = [idx for idx, _ in hits]

        # ¿Rerank?
        if model_cfg["pipeline"].get("reranker") == "cross_encoder":
            k_final = int(params.get("k", 3))
            reranked = rerank_cross_encoder(query, candidate_indices, top_k=k_final)

            retrieved = []
            for idx, score in reranked:
                retrieved.append({
                    "idx": idx,
                    "score": score,   # score del reranker
                    "chunk_id": chunk_ids[idx],
                    "chunk_ref": chunk_refs[idx],
                    "chunk_text": chunk_texts[idx]
                })
            return retrieved, time.time() - t0

        # Sin rerank
        retrieved = []
        for idx, score in hits[:k]:
            retrieved.append({
                "idx": idx,
                "score": score,
                "chunk_id": chunk_ids[idx],
                "chunk_ref": chunk_refs[idx],
                "chunk_text": chunk_texts[idx]
            })
        return retrieved, time.time() - t0

    # ----------------------------
    # Hybrid
    # ----------------------------
    if retriever == "hybrid":
        k = int(params.get("k", 3))
        alpha = float(params.get("alpha", 0.2))

        q_emb = embed_query_st(query)
        hits = hybrid_retrieve(
            bm25=bm25_index,
            query=query,
            query_emb=q_emb,
            doc_embs=embeddings,
            k=k,
            alpha=alpha
        )

        retrieved = []
        for idx, score in hits:
            retrieved.append({
                "idx": idx,
                "score": score,
                "chunk_id": chunk_ids[idx],
                "chunk_ref": chunk_refs[idx],
                "chunk_text": chunk_texts[idx]
            })
        return retrieved, time.time() - t0

    # ----------------------------
    # Dense OpenAI (placeholder por ahora)
    # ----------------------------
    if retriever == "dense_openai":
        # Este se implementa en Parte 8/9 cuando decidamos si lo corremos por costo
        raise NotImplementedError("dense_openai retrieval se implementará más adelante (M5).")

    raise ValueError(f"Retriever desconocido: {retriever}")

In [14]:
# ==========================================
# 6.4 - Runner unificado (run_model() unificado (caching + tiempos + output estándar))
# ==========================================

def run_model(model_cfg: dict, query: str, use_cache: bool = True):
    """
    Ejecuta un modelo sobre una query:
    - Retrieval (si aplica)
    - Construcción de contexto
    - Generación LLM
    - Registro de tiempos
    - Cache a disco

    Retorna: dict estandarizado
    """
    model_id = model_cfg["model_id"]
    cpath = cache_path(model_id, query)

    # ----------------------------
    # Cache hit
    # ----------------------------
    if use_cache and cpath.exists():
        with open(cpath, "r", encoding="utf-8") as f:
            return json.load(f)

    t_total0 = time.time()

    # ----------------------------
    # Retrieval
    # ----------------------------
    retrieved, t_retrieval = retrieve_for_model(model_cfg, query)

    # Construir contexto si hay retrieval
    if len(retrieved) > 0:
        context_str, citations = build_context(retrieved, max_chars=5500)
        user_prompt = model_cfg["prompt"]["user_template"].format(
            query=query,
            context=context_str
        )
    else:
        citations = []
        user_prompt = model_cfg["prompt"]["user_template"].format(query=query, context="")

    # ----------------------------
    # Generación
    # ----------------------------
    sys_prompt = model_cfg["prompt"]["system"]
    params = model_cfg["params"]

    answer, usage, t_gen = call_llm(
        system_prompt=sys_prompt,
        user_prompt=user_prompt,
        temperature=float(params.get("temperature", 0.2)),
        top_p=float(params.get("top_p", 1.0)),
        max_tokens=int(params.get("max_tokens", 500))
    )

    t_total = time.time() - t_total0

    # ----------------------------
    # Output estándar
    # ----------------------------
    out = {
        "model_id": model_id,
        "query": query,
        "retrieved": retrieved,     # incluye refs y texto
        "citations_from_context": citations,
        "answer": answer,
        "usage": usage,
        "timing": {
            "retrieval_s": t_retrieval,
            "generation_s": t_gen,
            "total_s": t_total
        },
        "meta": {
            "gen_model": GEN_MODEL_NAME,
            "timestamp": datetime.now().isoformat()
        }
    }

    # ----------------------------
    # Guardar cache
    # ----------------------------
    if use_cache:
        with open(cpath, "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)

    return out

print("run_model() listo.")

run_model() listo.


In [15]:
# ==========================================
# 6.5 - Smoke test rápido (1 query × 2 modelos, para validar pipeline)
# ==========================================

test_query = df_eval.iloc[0]["query"]

# Probar un modelo RAG y uno LLM-only
out_llm = run_model(MODELS["M1_LLM_ONLY"], test_query, use_cache=True)
out_dense = run_model(MODELS["M3_DENSE_ST_RAG"], test_query, use_cache=True)

print("Smoke test completado.")
print("\n--- M1_LLM_ONLY ---")
print("Tiempo total (s):", out_llm["timing"]["total_s"])
print("Respuesta (preview):", out_llm["answer"][:300], "...\n")

print("\n--- M3_DENSE_ST_RAG ---")
print("Tiempo total (s):", out_dense["timing"]["total_s"])
print("Citas (context):", out_dense["citations_from_context"][:5])
print("Respuesta (preview):", out_dense["answer"][:300], "...")

Smoke test completado.

--- M1_LLM_ONLY ---
Tiempo total (s): 15.172749042510986
Respuesta (preview): Querido hermano o hermana en Cristo, lamento profundamente tu pérdida y entiendo que este es un momento de gran dolor y confusión. La Biblia nos enseña que Dios está cerca de los quebrantados de corazón y que Él se preocupa por nosotros en nuestros momentos de aflicción. Aunque puede parecer que Dio ...


--- M3_DENSE_ST_RAG ---
Tiempo total (s): 17.269647121429443
Citas (context): ['ROM 9:1-3', 'PHP 2:25-27', 'JOB 6:9-10']
Respuesta (preview): Querido hermano o hermana en Cristo,

Lamento profundamente la pérdida de tu ser querido. El duelo es una experiencia dolorosa y, a menudo, puede hacernos sentir solos y abandonados. La Biblia nos muestra que incluso los más grandes siervos de Dios experimentaron tristeza y angustia. Por ejemplo, el ...


### Interpretación (Parte 6)

- Se implementó un `runner` unificado `run_model()` capaz de ejecutar cualquier configuración de `MODELS`.
- El runner:
  - Ejecuta retrieval según el modelo (si aplica).
  - Construye un contexto trazable con identificadores y referencias.
  - Llama al LLM con prompt estructurado.
  - Registra tiempos de retrieval, generación y total.
  - Almacena resultados en cache para evitar consumo innecesario de tokens.

- Se realizó un smoke test con:
  - Un modelo **LLM-only**
  - Un modelo **Dense-RAG**
  confirmando que el pipeline genera respuestas y puede recuperar contexto bíblico cuando corresponde.

Con esto, el proyecto queda listo para ejecutar evaluación batch de los 6 modelos sobre el Eval Set v1.

---

> *"Secóse la hierba, cayóse la flor; mas la palabra del Dios nuestro permanece para siempre."*
>
> — Isaías 40:8

---

# **PARTE 7 – Evaluación batch y comparación de modelos**

En esta sección se evalúan los 6 modelos alternativos sobre el Eval Set v1 (30 queries).

Se calculan:

### Métrica principal
- **Judge Score promedio** (evaluación automática mediante LLM-juez)

### Métricas adicionales
- Recall@k
- MRR (Mean Reciprocal Rank)
- Faithfulness (alineación con contexto recuperado)

### Métricas de eficiencia
- Tiempo promedio de retrieval
- Tiempo promedio de generación
- Tiempo total promedio

Los resultados se consolidan en una tabla comparativa ordenada por la métrica principal.

In [16]:
# ==========================================
# 7.1 - Métricas IR: Recall@k y MRR
# ==========================================

def recall_at_k(retrieved_refs, expected_refs, k):
    """
    retrieved_refs: lista de referencias recuperadas (ordenadas)
    expected_refs: lista de referencias esperadas
    """
    if not expected_refs:
        return None

    retrieved_topk = retrieved_refs[:k]
    hits = 0
    for er in expected_refs:
        if any(er.split()[0] in r for r in retrieved_topk):
            hits += 1

    return hits / len(expected_refs)


def mrr_score(retrieved_refs, expected_refs):
    """
    Mean Reciprocal Rank
    """
    if not expected_refs:
        return None

    for rank, ref in enumerate(retrieved_refs, start=1):
        if any(er.split()[0] in ref for er in expected_refs):
            return 1.0 / rank

    return 0.0

In [17]:
# ==========================================
# 7.2 - LLM-Judge (juez simplificado para evaluación automática)
# ==========================================

JUDGE_SYSTEM_PROMPT = """
Eres un evaluador experto en teología bíblica y sistemas RAG.
Evalúa de forma muy estricta la respuesta según:

1) Relevancia a la pregunta (0-5)
2) Fidelidad bíblica (0-5)
3) Calidad pastoral (0-5)

Devuelve solo un JSON:
{
  "relevance": int,
  "faithfulness": int,
  "pastoral": int,
  "overall": float
}
"""

def judge_response(query, answer):
    user_prompt = f"""
Pregunta:
{query}

Respuesta generada:
{answer}

Evalúa según los criterios.
"""
    text, usage, _ = call_llm(
        system_prompt=JUDGE_SYSTEM_PROMPT,
        user_prompt=user_prompt,
        temperature=0.0,
        max_tokens=200
    )

    try:
        import json
        scores = json.loads(text)
    except:
        scores = {
            "relevance": None,
            "faithfulness": None,
            "pastoral": None,
            "overall": None
        }

    return scores

In [18]:
# ==========================================
# 7.3 - Evaluación batch principal
# ==========================================

results = []

for model_id, model_cfg in MODELS.items():

    print(f"\nEvaluando {model_id}...")
    model_metrics = {
        "model_id": model_id,
        "judge_overall": [],
        "recall_at_3": [],
        "mrr": [],
        "retrieval_time": [],
        "generation_time": [],
        "total_time": []
    }

    for _, row in df_eval.iterrows():

        query = row["query"]
        expected_refs = row["expected_refs"]

        try:
            out = run_model(model_cfg, query, use_cache=True)
        except NotImplementedError:
            continue  # saltamos M5 por ahora

        retrieved_refs = [r["chunk_ref"] for r in out["retrieved"]]

        # Métricas IR
        r_at3 = recall_at_k(retrieved_refs, expected_refs, k=3)
        mrr = mrr_score(retrieved_refs, expected_refs)

        # Judge
        judge_scores = judge_response(query, out["answer"])

        # Guardar métricas
        model_metrics["judge_overall"].append(judge_scores.get("overall"))
        model_metrics["recall_at_3"].append(r_at3)
        model_metrics["mrr"].append(mrr)
        model_metrics["retrieval_time"].append(out["timing"]["retrieval_s"])
        model_metrics["generation_time"].append(out["timing"]["generation_s"])
        model_metrics["total_time"].append(out["timing"]["total_s"])

    # Promedios
    results.append({
        "model_id": model_id,
        "judge_overall_mean": np.nanmean(model_metrics["judge_overall"]),
        "recall_at_3_mean": np.nanmean(model_metrics["recall_at_3"]),
        "mrr_mean": np.nanmean(model_metrics["mrr"]),
        "retrieval_time_mean": np.nanmean(model_metrics["retrieval_time"]),
        "generation_time_mean": np.nanmean(model_metrics["generation_time"]),
        "total_time_mean": np.nanmean(model_metrics["total_time"])
    })

results_df = pd.DataFrame(results).sort_values(
    by="judge_overall_mean", ascending=False
)

results_df


Evaluando M1_LLM_ONLY...

Evaluando M2_BM25_RAG...

Evaluando M3_DENSE_ST_RAG...

Evaluando M4_HYBRID_RAG...

Evaluando M5_DENSE_OAI_RAG...

Evaluando M6_DENSE_ST_RERANK...


,model_id,judge_overall_mean,recall_at_3_mean,mrr_mean,retrieval_time_mean,generation_time_mean,total_time_mean
0,M1_LLM_ONLY,4.989000,0.000000,0.000000,5.801519e-07,6.406257,6.406337
5,M6_DENSE_ST_RERANK,4.846000,0.250000,0.416667,6.170928e-02,8.520052,8.581833
3,M4_HYBRID_RAG,4.768333,0.144444,0.211111,5.560583e-02,7.587177,7.642851
2,M3_DENSE_ST_RAG,4.768000,0.219444,0.394444,3.906722e-02,7.667524,7.706658
1,M2_BM25_RAG,4.690000,0.144444,0.216667,4.471872e-02,6.692512,6.737294
4,M5_DENSE_OAI_RAG,NaN,NaN,NaN,NaN,NaN,NaN


### Interpretación de resultados (Parte 7 - Evaluación inicial / Naive)

La tabla comparativa muestra el desempeño promedio de cada modelo bajo un esquema de evaluación **inicial** donde el juez (LLM-Judge) evalúa únicamente:

- La pregunta (`query`)
- La respuesta generada (`answer`)

Bajo este enfoque, puede sorprender que modelos **sin retrieval (LLM-only)** obtengan una puntuación más alta, ya que el juez puede premiar aspectos como:

- Tono pastoral
- Claridad
- Estructura

sin poder verificar si las afirmaciones o citas están **grounded** en pasajes recuperados.

Además, en modelos RAG el desempeño percibido puede verse afectado si el retriever recupera pasajes subóptimos para una consulta, pues el modelo intenta ser fiel al contexto entregado.

**Conclusión metodológica:** esta primera evaluación es útil como baseline comparativo, pero no mide de manera estricta el valor principal de RAG (grounding + trazabilidad).

Por ello, en la siguiente sección se implementa una evaluación **estricta y trazable** que:
- Extrae citas bíblicas mencionadas en la respuesta,
- Verifica cuáles aparecen realmente en el contexto recuperado,
- Penaliza automáticamente citas fuera de contexto (citation leakage),
- Y utiliza un juez que recibe también el contexto para evaluar faithfulness.

---

> *"El que habla la verdad da un testimonio justo, pero el testigo falso dice mentiras. Hay quienes hablan sin pensar y sus palabras hieren como espadas, pero la lengua de los sabios trae sanidad."*
>
> — Proverbios 12:17-18

---

# **PARTE 7B – Evaluación estricta y trazable (Grounded Judge + penalización por citas fuera de contexto)**

En la Parte 7 se realizó una evaluación inicial ("naive") donde el juez solo observa la pregunta y la respuesta.

Sin embargo, en sistemas RAG el valor principal es la **trazabilidad**:
- que la respuesta esté **grounded** en pasajes recuperados,
- y que las citas bíblicas usadas correspondan realmente al contexto entregado.

Por ello, en esta sección se implementa una evaluación más estricta que:

1) Extrae referencias bíblicas mencionadas en la respuesta (p. ej., "Romanos 8:38-39", "ROM 8:38-39", "Salmo 23").
2) Verifica si esas referencias aparecen en el conjunto de pasajes recuperados.
3) Calcula tasa de "citation leakage" y penaliza automáticamente el puntaje final.
4) Usa un juez grounded que recibe también el contexto recuperado.
5) Implementa el modelo M5 (Dense-RAG con embeddings OpenAI) y lo incluye en la comparación final.

In [19]:
# ==========================================
# Parte 7B.1 - Extractor de citas + normalización + verificación
# ==========================================

import re

# Mapeo mínimo (ampliable) de nombres en español/inglés -> códigos tipo dataset (ROM, MAT, PSA, etc.)
BOOK_MAP = {
    # Nuevo Testamento
    "mateo": "MAT", "mt": "MAT", "mat": "MAT",
    "marcos": "MRK", "mc": "MRK", "mrk": "MRK",
    "lucas": "LUK", "lc": "LUK", "luk": "LUK",
    "juan": "JHN", "jn": "JHN", "jhn": "JHN",
    "hechos": "ACT", "act": "ACT",
    "romanos": "ROM", "rom": "ROM",
    "corintios": "COR", "co": "COR",  # ojo: requiere prefijo 1/2 para precisión
    "1 corintios": "1COR", "1cor": "1COR",
    "2 corintios": "2COR", "2cor": "2COR",
    "gálatas": "GAL", "galatas": "GAL", "gal": "GAL",
    "efesios": "EPH", "ef": "EPH", "eph": "EPH",
    "filipenses": "PHP", "fil": "PHP", "php": "PHP",
    "colosenses": "COL", "col": "COL",
    "tesalonicenses": "TH",  # requiere 1/2
    "1 tesalonicenses": "1TH", "1th": "1TH",
    "2 tesalonicenses": "2TH", "2th": "2TH",
    "timoteo": "TIM",
    "1 timoteo": "1TIM", "1tim": "1TIM",
    "2 timoteo": "2TIM", "2tim": "2TIM",
    "tito": "TIT", "tit": "TIT",
    "hebreos": "HEB", "heb": "HEB",
    "santiago": "JAS", "jas": "JAS",
    "pedro": "PET",
    "1 pedro": "1PET", "1pet": "1PET",
    "2 pedro": "2PET", "2pet": "2PET",
    "juan 1": "1JHN", "1 juan": "1JHN", "1jn": "1JHN", "1jhn": "1JHN",
    "juan 2": "2JHN", "2 juan": "2JHN", "2jn": "2JHN", "2jhn": "2JHN",
    "juan 3": "3JHN", "3 juan": "3JHN", "3jn": "3JHN", "3jhn": "3JHN",
    "apocalipsis": "REV", "rev": "REV",

    # Antiguo Testamento (ejemplos más frecuentes)
    "salmo": "PSA", "salmos": "PSA", "ps": "PSA", "psa": "PSA",
    "proverbios": "PRO", "pro": "PRO",
    "job": "JOB", "jó": "JOB", "job.": "JOB",
    "isaías": "ISA", "isaias": "ISA", "isa": "ISA",
    "jeremías": "JER", "jeremias": "JER", "jer": "JER",
    "éxodo": "EXO", "exodo": "EXO", "exo": "EXO",
    "génesis": "GEN", "genesis": "GEN", "gen": "GEN",
    "1 reyes": "1KGS", "1reyes": "1KGS", "1kgs": "1KGS",
    "2 reyes": "2KGS", "2reyes": "2KGS", "2kgs": "2KGS",
}

# Regex para detectar patrones tipo:
# - ROM 8:38-39
# - Romanos 8:38-39
# - Salmo 23
# - 1 Corintios 13:4-7
# Nota: no pretendemos cubrir 100% de variantes, solo robusto para evaluación.
REF_REGEX = re.compile(
    r'(?P<prefix>\b[1-3]\s*)?'
    r'(?P<book>[A-Za-zÁÉÍÓÚÜÑáéíóúüñ\.]{2,20})'
    r'\s*'
    r'(?P<chapter>\d{1,3})'
    r'(?:\s*:\s*(?P<verse>\d{1,3})(?:\s*-\s*(?P<verse_end>\d{1,3}))?)?',
    flags=re.IGNORECASE
)

def normalize_book(prefix: str, book_raw: str) -> str | None:
    """
    Convierte 'Romanos'->ROM, 'Salmo'->PSA, '1 Corintios'->1COR, etc.
    """
    if book_raw is None:
        return None

    b = book_raw.strip().lower().replace(".", "")
    p = (prefix or "").strip().lower().replace(".", "")

    # Caso: códigos ya tipo ROM/MAT/PSA en texto
    if len(b) in (3,4,5) and b.isalpha():
        # ej: "rom" -> ROM
        code = b.upper()
        # si hay prefijo y el code es genérico tipo COR/TIM/PET/TH, intentamos expandir
        if p and code in ("COR", "TIM", "PET", "TH"):
            return f"{p.strip()}".replace(" ", "") .upper() + code
        return code

    # Caso: nombre tipo "corintios" con prefijo
    key = (p + " " + b).strip()
    if key in BOOK_MAP:
        return BOOK_MAP[key]

    # Caso: nombre sin prefijo
    if b in BOOK_MAP:
        code = BOOK_MAP[b]
        # si el code requiere prefijo pero no lo tenemos, devolvemos el code tal cual (mejor que None)
        return code

    return None

def parse_chunk_ref(ref: str):
    """
    Parse simple de refs tipo 'ROM 9:1-3' -> ('ROM', 9)
    """
    try:
        parts = ref.strip().split()
        book = parts[0].upper()
        chap = int(parts[1].split(":")[0])
        return book, chap
    except Exception:
        return None, None

def extract_citations(text: str):
    """
    Extrae citas como tuplas normalizadas: (book_code, chapter)
    """
    citations = []
    for m in REF_REGEX.finditer(text or ""):
        prefix = m.group("prefix")
        book_raw = m.group("book")
        chapter = m.group("chapter")

        book_code = normalize_book(prefix, book_raw)
        if book_code is None:
            continue

        try:
            chap = int(chapter)
        except:
            continue

        citations.append((book_code, chap))

    # unique preserving order
    seen = set()
    uniq = []
    for c in citations:
        if c not in seen:
            uniq.append(c)
            seen.add(c)
    return uniq

def citations_in_context(answer_citations, retrieved_chunk_refs):
    """
    Define "en contexto" si (book,chapter) de la cita aparece en algún chunk recuperado.
    """
    ctx = [parse_chunk_ref(r) for r in retrieved_chunk_refs]
    ctx_set = set([x for x in ctx if x[0] is not None])
    in_ctx, out_ctx = [], []
    for c in answer_citations:
        if c in ctx_set:
            in_ctx.append(c)
        else:
            out_ctx.append(c)
    return in_ctx, out_ctx

print("Extractor de citas + normalización listo.")

Extractor de citas + normalización listo.


In [20]:
# ==========================================
# 7B.2 - Judge grounded + penalización automática
# ==========================================

JUDGE_STRICT_SYSTEM = """
Eres un evaluador (judge) estricto para un asistente bíblico/pastoral con RAG.
Evalúa la respuesta considerando el CONTEXTO proporcionado.

Devuelve SOLO un JSON válido con llaves:
{
  "relevance": int,          # 1-5
  "faithfulness": int,       # 1-5 (debe estar respaldado por el contexto)
  "citation_accuracy": int,  # 1-5 (no inventar citas fuera del contexto)
  "pastoral": int,           # 1-5
  "overall": float,          # 1-5 (promedio ponderado)
  "comments": string         # breve
}

Reglas:
- Si la respuesta menciona referencias bíblicas que NO aparecen en el contexto recuperado, penaliza citation_accuracy y faithfulness.
- Premia respuestas que usan explícitamente el contexto recuperado y que son prudentes al afirmar.
"""

JUDGE_CACHE_DIR = BASE_PATH / "cache_judge_strict"
JUDGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def judge_cache_path(model_id: str, query: str) -> Path:
    h = _stable_hash(query)
    return JUDGE_CACHE_DIR / f"{model_id}__{h}.json"

def judge_strict(query: str, answer: str, context_str: str, retrieved_refs: list[str], use_cache: bool = True):
    """
    Judge grounded: ve query + answer + contexto + refs recuperadas.
    Cachea salida para no re-gastar tokens.
    """
    cpath = judge_cache_path("STRICT_JUDGE", query + "||" + answer[:200])  # más robusto
    if use_cache and cpath.exists():
        with open(cpath, "r", encoding="utf-8") as f:
            return json.load(f)

    # limitar contexto para judge (evitar tokens excesivos)
    ctx_short = (context_str or "")[:3500]
    refs_short = ", ".join(retrieved_refs[:10])

    user_prompt = f"""
PREGUNTA:
{query}

REFERENCIAS RECUPERADAS (subset):
{refs_short}

CONTEXTO RECUPERADO (truncado):
{ctx_short}

RESPUESTA A EVALUAR:
{answer}

Devuelve SOLO el JSON.
"""

    text, usage, t = call_llm(
        system_prompt=JUDGE_STRICT_SYSTEM,
        user_prompt=user_prompt,
        temperature=0.0,
        max_tokens=220
    )

    try:
        scores = json.loads(text)
    except Exception:
        scores = {
            "relevance": None,
            "faithfulness": None,
            "citation_accuracy": None,
            "pastoral": None,
            "overall": None,
            "comments": "Judge output no parseable."
        }

    scores["_judge_usage"] = usage
    scores["_judge_time_s"] = t

    if use_cache:
        with open(cpath, "w", encoding="utf-8") as f:
            json.dump(scores, f, ensure_ascii=False, indent=2)

    return scores

def apply_leakage_penalty(scores: dict, n_out: int):
    """
    Penalización determinística adicional (post-proceso) por leakage detectado.
    Esto hace la evaluación más trazable y menos dependiente de variación del judge.
    """
    if scores is None:
        return scores

    # Copia
    s = dict(scores)

    # Si no hay leakage, no tocar
    if n_out <= 0:
        s["overall_adjusted"] = s.get("overall")
        s["leakage_penalty"] = 0.0
        return s

    # Penalización: 0.4 por cita fuera de contexto, cap 1.6
    penalty = min(1.6, 0.4 * n_out)

    def clip(x):
        if x is None:
            return None
        return float(max(1.0, min(5.0, x)))

    # Ajustes conservadores
    s["faithfulness"] = clip((s.get("faithfulness") or 1.0) - min(2.0, 0.5 * n_out))
    s["citation_accuracy"] = clip((s.get("citation_accuracy") or 1.0) - min(3.0, 0.8 * n_out))

    overall = s.get("overall")
    if overall is None:
        overall = 1.0
    s["overall_adjusted"] = clip(overall - penalty)
    s["leakage_penalty"] = penalty

    return s

print("Judge strict + penalización automática listos.")

Judge strict + penalización automática listos.


In [21]:
# ==========================================
# 7B.3 - M5: Embeddings OpenAI + cache
# ==========================================

OAI_EMB_MODEL = "text-embedding-3-small"
OAI_EMB_CACHE_PATH = BASE_PATH / "openai_chunk_embeddings__v1.npz"

def l2_normalize(mat: np.ndarray, axis: int = 1, eps: float = 1e-12):
    norm = np.linalg.norm(mat, axis=axis, keepdims=True)
    norm = np.maximum(norm, eps)
    return mat / norm

def embed_openai_texts(texts: list[str], model: str = OAI_EMB_MODEL):
    """
    Embeddings OpenAI para lista de textos.
    """
    resp = client.embeddings.create(model=model, input=texts)
    embs = np.array([d.embedding for d in resp.data], dtype=np.float32)
    return embs

def build_or_load_openai_chunk_embeddings(force_rebuild: bool = False, batch_size: int = 128):
    """
    Genera embeddings para TODOS los chunks y cachea en .npz.
    """
    if OAI_EMB_CACHE_PATH.exists() and not force_rebuild:
        npz = np.load(OAI_EMB_CACHE_PATH, allow_pickle=True)
        emb = npz["emb"]
        cid = npz["chunk_id"]
        print(f"Cargado cache OpenAI chunk embeddings: {OAI_EMB_CACHE_PATH}")
        print("Shape:", emb.shape)
        return cid, emb

    print("Generando embeddings OpenAI para chunks (esto puede tardar)...")
    all_embs = []
    n = len(chunk_texts)

    t0 = time.time()
    for i in range(0, n, batch_size):
        batch = [str(x) for x in chunk_texts[i:i+batch_size]]
        embs = embed_openai_texts(batch, model=OAI_EMB_MODEL)
        all_embs.append(embs)

        if (i // batch_size) % 10 == 0:
            print(f"  - Progreso: {i}/{n}")

    emb = np.vstack(all_embs)
    emb = l2_normalize(emb, axis=1)  # normalizamos para cosine=dot

    elapsed = time.time() - t0
    np.savez(OAI_EMB_CACHE_PATH, chunk_id=chunk_ids, emb=emb, model=OAI_EMB_MODEL)

    print(f"Cache guardado en: {OAI_EMB_CACHE_PATH}")
    print("Shape:", emb.shape)
    print(f"Tiempo total embeddings chunks: {elapsed/60:.2f} min")

    return chunk_ids, emb

# Construir/cargar embeddings de chunks para M5
oai_chunk_ids, oai_chunk_embs = build_or_load_openai_chunk_embeddings(force_rebuild=False, batch_size=128)

# Sanity: alineación esperada con chunk_ids
assert np.array_equal(oai_chunk_ids, chunk_ids), "OpenAI chunk embeddings no están alineados con df_chunks."
print("M5 listo: embeddings OpenAI de chunks cargados y alineados.")

Cargado cache OpenAI chunk embeddings: /content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/Avance 4 - Modelos alternativos/Avance 4 - Steven/openai_chunk_embeddings__v1.npz
Shape: (10058, 1536)
M5 listo: embeddings OpenAI de chunks cargados y alineados.


In [22]:
# ==========================================
# 7B.4 - Retrieval y runner M5
# ==========================================

M5_CACHE_DIR = BASE_PATH / "cache_m5"
M5_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def m5_cache_path(query: str) -> Path:
    h = _stable_hash(query)
    return M5_CACHE_DIR / f"M5__{h}.json"

def embed_query_openai(query: str):
    emb = embed_openai_texts([query], model=OAI_EMB_MODEL)[0]
    emb = emb / (np.linalg.norm(emb) + 1e-12)
    return emb.astype(np.float32)

def dense_retrieve_openai(query_emb: np.ndarray, doc_embs: np.ndarray, k: int = 3):
    sims = np.dot(query_emb.reshape(1,-1), doc_embs.T).flatten()
    top_idx = np.argsort(sims)[::-1][:k]
    return [(int(i), float(sims[i])) for i in top_idx]

def run_model_m5(model_cfg: dict, query: str, use_cache: bool = True):
    """
    Runner específico para M5, manteniendo el mismo output estándar de run_model().
    """
    cpath = m5_cache_path(query)
    if use_cache and cpath.exists():
        with open(cpath, "r", encoding="utf-8") as f:
            return json.load(f)

    t_total0 = time.time()

    # Retrieval
    t0 = time.time()
    q_emb = embed_query_openai(query)
    k = int(model_cfg["params"].get("k", 3))
    hits = dense_retrieve_openai(q_emb, oai_chunk_embs, k=k)

    retrieved = []
    for idx, score in hits:
        retrieved.append({
            "idx": idx,
            "score": score,
            "chunk_id": chunk_ids[idx],
            "chunk_ref": chunk_refs[idx],
            "chunk_text": chunk_texts[idx]
        })
    t_retrieval = time.time() - t0

    # Context
    context_str, citations = build_context(retrieved, max_chars=5500)
    user_prompt = model_cfg["prompt"]["user_template"].format(query=query, context=context_str)

    # Generación
    sys_prompt = model_cfg["prompt"]["system"]
    params = model_cfg["params"]

    answer, usage, t_gen = call_llm(
        system_prompt=sys_prompt,
        user_prompt=user_prompt,
        temperature=float(params.get("temperature", 0.2)),
        top_p=float(params.get("top_p", 1.0)),
        max_tokens=int(params.get("max_tokens", 500))
    )

    t_total = time.time() - t_total0

    out = {
        "model_id": model_cfg["model_id"],
        "query": query,
        "retrieved": retrieved,
        "citations_from_context": citations,
        "answer": answer,
        "usage": usage,
        "timing": {
            "retrieval_s": t_retrieval,
            "generation_s": t_gen,
            "total_s": t_total
        },
        "meta": {
            "gen_model": GEN_MODEL_NAME,
            "embedding_model": OAI_EMB_MODEL,
            "timestamp": datetime.now().isoformat()
        }
    }

    if use_cache:
        with open(cpath, "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)

    return out

print("Runner M5 listo.")

Runner M5 listo.


In [23]:
# ==========================================
# 7B.5 - Evaluando batch STRICT (Grounded + Leakage + M5)
# Nueva tabla comparativa
# ==========================================

def get_output_for_model(model_id: str, model_cfg: dict, query: str, use_cache: bool = True):
    """
    Reutiliza cache del runner original para M1/M2/M3/M4/M6.
    Usa runner especial para M5.
    """
    if model_id == "M5_DENSE_OAI_RAG":
        return run_model_m5(model_cfg, query, use_cache=use_cache)
    else:
        return run_model(model_cfg, query, use_cache=use_cache)

strict_rows = []

for model_id, model_cfg in MODELS.items():
    print(f"\n[STRICT] Evaluando {model_id}...")

    judge_overall_adj = []
    judge_relevance = []
    judge_faith = []
    judge_citation_acc = []
    judge_pastoral = []
    leakage_rate = []

    recall3_list = []
    mrr_list = []

    t_ret = []
    t_gen = []
    t_tot = []

    for _, row in df_eval.iterrows():
        query = row["query"]
        expected_refs = row["expected_refs"]

        out = get_output_for_model(model_id, model_cfg, query, use_cache=True)

        retrieved_refs = [r["chunk_ref"] for r in out["retrieved"]]
        context_str, _ = build_context(out["retrieved"], max_chars=3500)  # mismo formato, pero truncado

        # ---- IR metrics (igual que antes)
        r_at3 = recall_at_k(retrieved_refs, expected_refs, k=3)
        mrr = mrr_score(retrieved_refs, expected_refs)

        recall3_list.append(r_at3)
        mrr_list.append(mrr)

        # ---- Leakage
        ans_cits = extract_citations(out["answer"])
        _, out_ctx = citations_in_context(ans_cits, retrieved_refs)
        leak = 1.0 if len(out_ctx) > 0 else 0.0
        leakage_rate.append(leak)

        # ---- Judge grounded + penalización
        js = judge_strict(
            query=query,
            answer=out["answer"],
            context_str=context_str,
            retrieved_refs=retrieved_refs,
            use_cache=True
        )
        js2 = apply_leakage_penalty(js, n_out=len(out_ctx))

        judge_overall_adj.append(js2.get("overall_adjusted"))
        judge_relevance.append(js2.get("relevance"))
        judge_faith.append(js2.get("faithfulness"))
        judge_citation_acc.append(js2.get("citation_accuracy"))
        judge_pastoral.append(js2.get("pastoral"))

        # ---- timing
        t_ret.append(out["timing"]["retrieval_s"])
        t_gen.append(out["timing"]["generation_s"])
        t_tot.append(out["timing"]["total_s"])

    strict_rows.append({
        "model_id": model_id,
        "judge_overall_adj_mean": np.nanmean(judge_overall_adj),
        "relevance_mean": np.nanmean(judge_relevance),
        "faithfulness_mean": np.nanmean(judge_faith),
        "citation_accuracy_mean": np.nanmean(judge_citation_acc),
        "pastoral_mean": np.nanmean(judge_pastoral),
        "leakage_rate_mean": np.nanmean(leakage_rate),
        "recall_at_3_mean": np.nanmean(recall3_list),
        "mrr_mean": np.nanmean(mrr_list),
        "retrieval_time_mean": np.nanmean(t_ret),
        "generation_time_mean": np.nanmean(t_gen),
        "total_time_mean": np.nanmean(t_tot),
    })

strict_results_df = pd.DataFrame(strict_rows).sort_values(
    by="judge_overall_adj_mean", ascending=False
)

strict_results_df


[STRICT] Evaluando M1_LLM_ONLY...

[STRICT] Evaluando M2_BM25_RAG...

[STRICT] Evaluando M3_DENSE_ST_RAG...

[STRICT] Evaluando M4_HYBRID_RAG...

[STRICT] Evaluando M5_DENSE_OAI_RAG...

[STRICT] Evaluando M6_DENSE_ST_RERANK...


,model_id,judge_overall_adj_mean,relevance_mean,faithfulness_mean,citation_accuracy_mean,pastoral_mean,leakage_rate_mean,recall_at_3_mean,mrr_mean,retrieval_time_mean,generation_time_mean,total_time_mean
4,M5_DENSE_OAI_RAG,4.386667,5.000000,4.200000,3.780000,4.966667,0.733333,0.291667,0.422222,3.289374e-01,7.736776,8.065791
2,M3_DENSE_ST_RAG,4.306667,5.000000,4.033333,3.700000,4.966667,0.733333,0.219444,0.394444,3.906722e-02,7.667524,7.706658
5,M6_DENSE_ST_RERANK,4.263333,4.966667,4.000000,3.580000,4.966667,0.800000,0.250000,0.416667,6.170928e-02,8.520052,8.581833
1,M2_BM25_RAG,4.160000,4.966667,3.866667,3.346667,4.933333,0.800000,0.144444,0.216667,4.471872e-02,6.692512,6.737294
3,M4_HYBRID_RAG,4.155000,5.000000,3.816667,3.340000,4.933333,0.800000,0.144444,0.211111,5.560583e-02,7.587177,7.642851
0,M1_LLM_ONLY,3.893333,5.000000,3.600000,2.786667,4.933333,1.000000,0.000000,0.000000,5.801519e-07,6.406257,6.406337


### Interpretación (Parte 7B - Evaluación estricta y trazable)

La evaluación estricta introduce verificación explícita de grounding:

- El juez ahora observa el contexto recuperado.
- Se detectan citas bíblicas mencionadas en la respuesta.
- Se penalizan automáticamente referencias fuera de contexto (citation leakage).

Bajo este esquema:

1. El modelo LLM-only cae al último lugar debido a:
   - Leakage rate = 1.0
   - Baja precisión de citas
   - Ausencia de grounding verificable.

2. Los modelos RAG superan claramente a LLM-only cuando se evalúa fidelidad y trazabilidad.

3. M5 (Dense-RAG con embeddings OpenAI) obtiene el mejor desempeño global:
   - Mayor Recall@3 y MRR.
   - Mejor balance entre relevancia, fidelidad y precisión de citas.
   - Menor leakage relativo dentro de los modelos evaluados.

4. M3 (Dense ST local) se mantiene competitivo, mostrando que embeddings locales también son viables.

5. El reranking (M6) mejora métricas IR, pero no reduce leakage generativo, lo cual confirma que el problema principal no es solo ranking sino control generativo.

Conclusión metodológica:
La evaluación estricta confirma que el valor real de RAG en LUMINA no es mejorar el tono pastoral (donde el LLM ya es fuerte), sino aumentar la trazabilidad y reducir alucinaciones bíblicas.

---

> *“Lo he llenado del Espíritu de Dios, en sabiduría, en inteligencia, en ciencia y en todo arte, para inventar diseños…”*
>
> Éxodo 31:3-5

---

# **PARTE 8 – Selección de los 2 mejores modelos**

Con base en la evaluación estricta y trazable (Parte 7B), se seleccionan los dos modelos con mejor desempeño global (`judge_overall_adj_mean`) considerando también:

- Faithfulness y citation_accuracy (grounding)
- Leakage rate (citas fuera de contexto)
- Métricas IR (Recall@3, MRR)
- Eficiencia (tiempos promedio)
- Complejidad y reproducibilidad del pipeline

La selección Top-2 permite enfocar el ajuste de hiperparámetros (Parte 9) únicamente en los modelos más prometedores.

In [24]:
# ==========================================
# Parte 8 - Selección Top-2 (STRICT) y tabla de resumen
# ==========================================

# Tomamos la tabla strict ya calculada
df_strict = strict_results_df.copy()

# Selección Top-2 por métrica principal strict
top2 = df_strict.sort_values("judge_overall_adj_mean", ascending=False).head(2)

print("Top-2 modelos seleccionados (STRICT):")
display(top2)

TOP2_MODEL_IDS = top2["model_id"].tolist()
TOP2_MODEL_IDS

Top-2 modelos seleccionados (STRICT):


,model_id,judge_overall_adj_mean,relevance_mean,faithfulness_mean,citation_accuracy_mean,pastoral_mean,leakage_rate_mean,recall_at_3_mean,mrr_mean,retrieval_time_mean,generation_time_mean,total_time_mean
4,M5_DENSE_OAI_RAG,4.386667,5.0,4.200000,3.78,4.966667,0.733333,0.291667,0.422222,0.328937,7.736776,8.065791
2,M3_DENSE_ST_RAG,4.306667,5.0,4.033333,3.70,4.966667,0.733333,0.219444,0.394444,0.039067,7.667524,7.706658


['M5_DENSE_OAI_RAG', 'M3_DENSE_ST_RAG']

### Justificación de selección Top-2 (basada en evaluación STRICT)

Los modelos seleccionados fueron:

1) **M5 – Dense OpenAI RAG**
2) **M3 – Dense ST RAG (embeddings locales SentenceTransformers)**

**Razones principales:**

- Ambos modelos obtuvieron los mayores valores de `judge_overall_adj_mean` bajo evaluación estricta, lo cual indica mejor equilibrio entre:
  - Relevancia,
  - Fidelidad al contexto recuperado (faithfulness),
  - Precisión de citas (citation_accuracy),
  - Tono pastoral.

- En métricas de recuperación (IR), M5 mostró el mejor desempeño (Recall@3 y MRR), sugiriendo que los embeddings OpenAI aportan una representación semántica más efectiva para el dominio bíblico en estas consultas.

- M3 se mantuvo altamente competitivo usando embeddings locales, lo cual aporta una alternativa reproducible e independiente de servicios externos, con menor complejidad operacional.

**Por qué no se seleccionaron los demás:**

- **M1 (LLM-only)**: aunque puede producir respuestas pastoralmente sólidas, bajo evaluación estricta presenta alta tasa de citas fuera de contexto (leakage) y ausencia de grounding verificable, lo cual contradice el objetivo central de un asistente RAG trazable.

- **M2 (BM25)** y **M4 (Hybrid)**: muestran desempeño inferior tanto en métricas IR como en score final estricto, sin aportar ventajas claras frente a Dense.

- **M6 (Rerank)**: aunque mejora métricas IR respecto a Dense básico, no reduce el leakage generativo, lo cual limita su ganancia bajo evaluación estricta.

Con base en lo anterior, el ajuste fino (Parte 9) se enfocará en M5 y M3 para identificar la configuración óptima.

In [25]:
# ==========================================
# 8.2 - Ejemplos cualitativos (2-3 queries) para visualización del lector
# ==========================================

def pretty_block(title: str):
    print("\n" + "="*90)
    print(title)
    print("="*90)

def format_retrieved_list(out, max_items=3):
    """
    Lista de chunks recuperados: ref + score + preview corto
    """
    items = out.get("retrieved", [])[:max_items]
    lines = []
    for r in items:
        ref = r.get("chunk_ref")
        score = r.get("score")
        txt = (r.get("chunk_text") or "").replace("\n", " ").strip()
        preview = (txt[:220] + "...") if len(txt) > 220 else txt
        lines.append(f"- {ref} | score={score:.4f}\n  {preview}")
    return "\n".join(lines) if lines else "(Sin retrieval)"

def format_citations(ans_cits, in_ctx, out_ctx):
    """
    Formatea citas detectadas y leakage.
    """
    def fmt(cits):
        return ", ".join([f"{b} {ch}" for (b, ch) in cits]) if cits else "—"

    return (
        f"Citas detectadas en respuesta: {fmt(ans_cits)}\n"
        f"✓ En contexto: {fmt(in_ctx)}\n"
        f"✗ Fuera de contexto: {fmt(out_ctx)}"
    )

def get_output_for_model_any(model_id: str, query: str, use_cache: bool = True):
    """
    Usa el runner correcto según modelo.
    """
    cfg = MODELS[model_id]
    if model_id == "M5_DENSE_OAI_RAG":
        return run_model_m5(cfg, query, use_cache=use_cache)
    else:
        return run_model(cfg, query, use_cache=use_cache)

# Elegimos 3 queries de ejemplo (puedes cambiar índices a mano si quieres)
example_idx = [0, 7, 18]  # 3 ejemplos variados del eval set
example_rows = df_eval.iloc[example_idx].to_dict(orient="records")

for ex_i, row in enumerate(example_rows, start=1):
    query = row["query"]
    expected_refs = row.get("expected_refs", [])

    pretty_block(f"EJEMPLO {ex_i} | Query")
    print(query)
    print("\nExpected refs (gold, si aplica):", expected_refs)

    # Para cada modelo Top-2, imprimimos salida completa legible
    for model_id in TOP2_MODEL_IDS:
        out = get_output_for_model_any(model_id, query, use_cache=True)

        retrieved_refs = [r["chunk_ref"] for r in out["retrieved"]]
        context_str, _ = build_context(out["retrieved"], max_chars=3500)

        # Leakage (extractor)
        ans_cits = extract_citations(out["answer"])
        in_ctx, out_ctx = citations_in_context(ans_cits, retrieved_refs)

        # Judge strict + penalización
        js = judge_strict(
            query=query,
            answer=out["answer"],
            context_str=context_str,
            retrieved_refs=retrieved_refs,
            use_cache=True
        )
        js2 = apply_leakage_penalty(js, n_out=len(out_ctx))

        pretty_block(f"{model_id} | Respuesta completa + grounding + evaluación (STRICT)")
        print("Tiempos (s):",
              f"retrieval={out['timing']['retrieval_s']:.3f}",
              f"gen={out['timing']['generation_s']:.3f}",
              f"total={out['timing']['total_s']:.3f}")

        print("\nTop recuperados:")
        print(format_retrieved_list(out, max_items=3))

        print("\nCitas y leakage:")
        print(format_citations(ans_cits, in_ctx, out_ctx))
        print(f"Leakage detected? {'Sí' if len(out_ctx)>0 else 'No'} (n_out={len(out_ctx)})")

        print("\nJudge STRICT (con penalización automática si hubo leakage):")
        print(f"- relevance:          {js2.get('relevance')}")
        print(f"- faithfulness:       {js2.get('faithfulness')}")
        print(f"- citation_accuracy:  {js2.get('citation_accuracy')}")
        print(f"- pastoral:           {js2.get('pastoral')}")
        print(f"- overall:            {js2.get('overall')}")
        print(f"- leakage_penalty:    {js2.get('leakage_penalty')}")
        print(f"- overall_adjusted:   {js2.get('overall_adjusted')}")
        if js2.get("comments"):
            print(f"- comments:           {js2.get('comments')}")

        print("\nRespuesta (texto completo):\n")
        print(out["answer"])


EJEMPLO 1 | Query
Perdí a un familiar y siento que Dios me abandonó. ¿Qué dice la Biblia para alguien en duelo y sin fuerzas?

Expected refs (gold, si aplica): ['PSA 23', 'MAT 5:4', 'JHN 11', 'ROM 8:38-39']

M5_DENSE_OAI_RAG | Respuesta completa + grounding + evaluación (STRICT)
Tiempos (s): retrieval=0.192 gen=8.598 total=8.790

Top recuperados:
- PSA 42:9-10 | score=0.5135
  Diré á Dios: Roca mía, ¿por qué te has olvidado de mí? ¿Por qué andaré yo enlutado por la opresión del enemigo? Mientras se están quebrantando mis huesos, mis enemigos me afrentan, diciéndome cada día: ¿Dónde está tu Dio...
- LAM 3:17-20 | score=0.5099
  Y mi alma se alejó de la paz, olvidéme del bien. Y dije: Pereció mi fortaleza, y mi esperanza de Jehová. Acuérdate de mi aflicción y de mi abatimiento, del ajenjo y de la hiel. Tendrálo aún en memoria mi alma, porque en ...
- JER 16:5-8 | score=0.5078
  Porque así ha dicho Jehová: No entres en casa de luto, ni vayas á lamentar, ni los consueles: porque yo he qui

---

> *“Y dediqué mi corazón a inquirir y a explorar con sabiduría todo lo que se hace bajo el cielo…”*
>
> Eclesiastés 1:13 –

---

# **PARTE 9 – Ajuste fino de los dos mejores modelos (Top-2)**

En esta sección se realiza ajuste fino (tuning) sobre los dos modelos seleccionados en Parte 8:

- M5 (Dense-RAG con embeddings OpenAI)
- M3 (Dense-RAG con embeddings locales SentenceTransformers)

Se exploran hiperparámetros con alto impacto y bajo costo computacional:

- `k` (número de pasajes recuperados): {1, 3, 5}
- `temperature` (control de variabilidad generativa): {0.0, 0.2, 0.5}
- `context_max_chars` (tamaño máximo de contexto incluido): {3000, 5500}

Para cada configuración se evalúa con el mismo esquema STRICT (Parte 7B):
- Judge grounded + penalización por leakage
- Métricas IR (Recall@k, MRR)
- Métricas de eficiencia (tiempos)

El objetivo es identificar la configuración que maximiza `judge_overall_adj_mean` manteniendo leakage controlado y tiempos razonables.

In [26]:
# ==========================================
# 9.2 - Runner tuning + cache por config
# ==========================================

TUNE_CACHE_DIR = BASE_PATH / "cache_tuning"
TUNE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def tune_cache_path(model_id: str, cfg_hash: str, query: str) -> Path:
    hq = _stable_hash(query)
    return TUNE_CACHE_DIR / f"{model_id}__{cfg_hash}__{hq}.json"

def hash_config(d: dict) -> str:
    s = json.dumps(d, sort_keys=True, ensure_ascii=False)
    return _stable_hash(s)

def run_model_tuned(model_id: str, query: str, k: int, temperature: float, context_max_chars: int, use_cache=True):
    """
    Ejecuta M3 o M5 con hiperparámetros tunables.
    Mantiene output estándar (similar a run_model).
    """
    assert model_id in ("M3_DENSE_ST_RAG", "M5_DENSE_OAI_RAG")
    base_cfg = MODELS[model_id]

    tuned_cfg = {
        "model_id": model_id,
        "k": int(k),
        "temperature": float(temperature),
        "context_max_chars": int(context_max_chars),
        "gen_model": GEN_MODEL_NAME
    }
    cfg_hash = hash_config(tuned_cfg)
    cpath = tune_cache_path(model_id, cfg_hash, query)

    if use_cache and cpath.exists():
        with open(cpath, "r", encoding="utf-8") as f:
            return json.load(f)

    t_total0 = time.time()

    # ---------- Retrieval ----------
    t0 = time.time()

    if model_id == "M3_DENSE_ST_RAG":
        q_emb = embed_query_st(query)
        hits = dense_retrieve(q_emb, embeddings, k=k)

        retrieved = []
        for idx, score in hits:
            retrieved.append({
                "idx": idx,
                "score": score,
                "chunk_id": chunk_ids[idx],
                "chunk_ref": chunk_refs[idx],
                "chunk_text": chunk_texts[idx]
            })

    elif model_id == "M5_DENSE_OAI_RAG":
        q_emb = embed_query_openai(query)
        hits = dense_retrieve_openai(q_emb, oai_chunk_embs, k=k)

        retrieved = []
        for idx, score in hits:
            retrieved.append({
                "idx": idx,
                "score": score,
                "chunk_id": chunk_ids[idx],
                "chunk_ref": chunk_refs[idx],
                "chunk_text": chunk_texts[idx]
            })

    t_retrieval = time.time() - t0

    # ---------- Context ----------
    context_str, citations = build_context(retrieved, max_chars=context_max_chars)
    user_prompt = base_cfg["prompt"]["user_template"].format(query=query, context=context_str)

    # ---------- Generation ----------
    sys_prompt = base_cfg["prompt"]["system"]
    answer, usage, t_gen = call_llm(
        system_prompt=sys_prompt,
        user_prompt=user_prompt,
        temperature=temperature,
        top_p=float(base_cfg["params"].get("top_p", 1.0)),
        max_tokens=int(base_cfg["params"].get("max_tokens", 500))
    )

    t_total = time.time() - t_total0

    out = {
        "model_id": model_id,
        "tuned_cfg": tuned_cfg,
        "query": query,
        "retrieved": retrieved,
        "citations_from_context": citations,
        "answer": answer,
        "usage": usage,
        "timing": {
            "retrieval_s": t_retrieval,
            "generation_s": t_gen,
            "total_s": t_total
        },
        "meta": {
            "cfg_hash": cfg_hash,
            "timestamp": datetime.now().isoformat()
        }
    }

    if use_cache:
        with open(cpath, "w", encoding="utf-8") as f:
            json.dump(out, f, ensure_ascii=False, indent=2)

    return out

print("Runner tuning listo (M3/M5). Cache:", TUNE_CACHE_DIR)

Runner tuning listo (M3/M5). Cache: /content/drive/MyDrive/Maestría ITESM/LUMINA - Proyecto Integrador/Avance 4 - Modelos alternativos/Avance 4 - Steven/cache_tuning


In [28]:
# ==========================================
# Parte 9.3 - Grid tuning REDUCIDO (subset) + evaluación STRICT
# ==========================================

# Subset de tuning (rápido y defendible)
TUNE_N = 10
df_tune = df_eval.sample(TUNE_N, random_state=42).reset_index(drop=True)

# Grid reducido
k_grid = [1, 3]
temp_grid = [0.0, 0.2]
ctx_grid = [3500]  # fijo para evitar explosión combinatoria

models_to_tune = ["M5_DENSE_OAI_RAG", "M3_DENSE_ST_RAG"]

tune_rows = []

for model_id in models_to_tune:
    for k in k_grid:
        for temp in temp_grid:
            for ctx_max in ctx_grid:

                judge_adj = []
                leakage = []
                recallk = []
                mrrs = []
                t_ret = []
                t_gen = []
                t_tot = []

                for _, row in df_tune.iterrows():
                    query = row["query"]
                    expected_refs = row.get("expected_refs", [])

                    out = run_model_tuned(
                        model_id=model_id,
                        query=query,
                        k=k,
                        temperature=temp,
                        context_max_chars=ctx_max,
                        use_cache=True
                    )

                    retrieved_refs = [r["chunk_ref"] for r in out["retrieved"]]
                    # Para el judge strict pasamos contexto truncado (controlado)
                    context_str, _ = build_context(out["retrieved"], max_chars=3500)

                    # IR metrics (adaptadas a k)
                    recallk.append(recall_at_k(retrieved_refs, expected_refs, k=k))
                    mrrs.append(mrr_score(retrieved_refs, expected_refs))

                    # Leakage
                    ans_cits = extract_citations(out["answer"])
                    _, out_ctx = citations_in_context(ans_cits, retrieved_refs)
                    leakage.append(1.0 if len(out_ctx) > 0 else 0.0)

                    # Judge strict + penalty
                    js = judge_strict(query, out["answer"], context_str, retrieved_refs, use_cache=True)
                    js2 = apply_leakage_penalty(js, n_out=len(out_ctx))
                    judge_adj.append(js2.get("overall_adjusted"))

                    # Timing
                    t_ret.append(out["timing"]["retrieval_s"])
                    t_gen.append(out["timing"]["generation_s"])
                    t_tot.append(out["timing"]["total_s"])

                tune_rows.append({
                    "model_id": model_id,
                    "k": k,
                    "temperature": temp,
                    "context_max_chars": ctx_max,
                    "judge_overall_adj_mean": np.nanmean(judge_adj),
                    "leakage_rate_mean": np.nanmean(leakage),
                    "recall_at_k_mean": np.nanmean(recallk),
                    "mrr_mean": np.nanmean(mrrs),
                    "retrieval_time_mean": np.nanmean(t_ret),
                    "generation_time_mean": np.nanmean(t_gen),
                    "total_time_mean": np.nanmean(t_tot),
                    "n_queries": TUNE_N
                })

tune_df = pd.DataFrame(tune_rows).sort_values(
    by=["judge_overall_adj_mean", "leakage_rate_mean"],
    ascending=[False, True]
)

print("Tuning grid REDUCIDO completado (subset). Todas las configs:")
display(tune_df)

Tuning grid REDUCIDO completado (subset). Todas las configs:


,model_id,k,temperature,context_max_chars,judge_overall_adj_mean,leakage_rate_mean,recall_at_k_mean,mrr_mean,retrieval_time_mean,generation_time_mean,total_time_mean,n_queries
0,M5_DENSE_OAI_RAG,1,0.0,3500,4.84,0.3,0.125000,0.30,0.298061,6.019401,6.317557,10
1,M5_DENSE_OAI_RAG,1,0.2,3500,4.80,0.3,0.125000,0.30,0.263720,6.039396,6.303212,10
4,M3_DENSE_ST_RAG,1,0.0,3500,4.68,0.5,0.058333,0.20,0.054527,7.197708,7.252323,10
3,M5_DENSE_OAI_RAG,3,0.2,3500,4.66,0.5,0.375000,0.55,0.192813,8.182436,8.375355,10
5,M3_DENSE_ST_RAG,1,0.2,3500,4.60,0.4,0.058333,0.20,0.060734,7.098436,7.159261,10
2,M5_DENSE_OAI_RAG,3,0.0,3500,4.52,0.6,0.375000,0.55,0.194673,7.643193,7.837978,10
7,M3_DENSE_ST_RAG,3,0.2,3500,4.46,0.6,0.125000,0.30,0.058388,8.286826,8.345327,10
6,M3_DENSE_ST_RAG,3,0.0,3500,4.38,0.7,0.125000,0.30,0.059907,9.148952,9.209167,10


In [29]:
# ==========================================
# 9.4 - Mejor config por modelo + validación full
# ==========================================

best_by_model = (
    tune_df.sort_values(["model_id", "judge_overall_adj_mean", "leakage_rate_mean"],
                        ascending=[True, False, True])
          .groupby("model_id")
          .head(1)
          .reset_index(drop=True)
)

print("Mejor configuración por modelo (según subset tuning):")
display(best_by_model)

def eval_config_full(model_id: str, k: int, temperature: float, context_max_chars: int):
    judge_adj = []
    leakage = []
    recallk = []
    mrrs = []
    t_ret = []
    t_gen = []
    t_tot = []

    for _, row in df_eval.iterrows():
        query = row["query"]
        expected_refs = row.get("expected_refs", [])

        out = run_model_tuned(model_id, query, k, temperature, context_max_chars, use_cache=True)
        retrieved_refs = [r["chunk_ref"] for r in out["retrieved"]]
        context_str, _ = build_context(out["retrieved"], max_chars=context_max_chars)

        recallk.append(recall_at_k(retrieved_refs, expected_refs, k=k))
        mrrs.append(mrr_score(retrieved_refs, expected_refs))

        ans_cits = extract_citations(out["answer"])
        _, out_ctx = citations_in_context(ans_cits, retrieved_refs)
        leakage.append(1.0 if len(out_ctx) > 0 else 0.0)

        js = judge_strict(query, out["answer"], context_str, retrieved_refs, use_cache=True)
        js2 = apply_leakage_penalty(js, n_out=len(out_ctx))
        judge_adj.append(js2.get("overall_adjusted"))

        t_ret.append(out["timing"]["retrieval_s"])
        t_gen.append(out["timing"]["generation_s"])
        t_tot.append(out["timing"]["total_s"])

    return {
        "model_id": model_id,
        "k": k,
        "temperature": temperature,
        "context_max_chars": context_max_chars,
        "judge_overall_adj_mean": np.nanmean(judge_adj),
        "leakage_rate_mean": np.nanmean(leakage),
        "recall_at_k_mean": np.nanmean(recallk),
        "mrr_mean": np.nanmean(mrrs),
        "retrieval_time_mean": np.nanmean(t_ret),
        "generation_time_mean": np.nanmean(t_gen),
        "total_time_mean": np.nanmean(t_tot),
        "n_queries": len(df_eval)
    }

full_rows = []
for _, r in best_by_model.iterrows():
    full_rows.append(
        eval_config_full(
            model_id=r["model_id"],
            k=int(r["k"]),
            temperature=float(r["temperature"]),
            context_max_chars=int(r["context_max_chars"])
        )
    )

full_validate_df = pd.DataFrame(full_rows).sort_values("judge_overall_adj_mean", ascending=False)

print("Validación FULL (30 queries) con mejor config por modelo:")
display(full_validate_df)

full_validate_df

Mejor configuración por modelo (según subset tuning):


,model_id,k,temperature,context_max_chars,judge_overall_adj_mean,leakage_rate_mean,recall_at_k_mean,mrr_mean,retrieval_time_mean,generation_time_mean,total_time_mean,n_queries
0,M3_DENSE_ST_RAG,1,0.0,3500,4.68,0.5,0.058333,0.2,0.054527,7.197708,7.252323,10
1,M5_DENSE_OAI_RAG,1,0.0,3500,4.84,0.3,0.125000,0.3,0.298061,6.019401,6.317557,10


Validación FULL (30 queries) con mejor config por modelo:


,model_id,k,temperature,context_max_chars,judge_overall_adj_mean,leakage_rate_mean,recall_at_k_mean,mrr_mean,retrieval_time_mean,generation_time_mean,total_time_mean,n_queries
1,M5_DENSE_OAI_RAG,1,0.0,3500,4.640,0.433333,0.097222,0.266667,0.289620,7.207142,7.496856,30
0,M3_DENSE_ST_RAG,1,0.0,3500,4.415,0.633333,0.075000,0.233333,0.063661,7.224708,7.288462,30


,model_id,k,temperature,context_max_chars,judge_overall_adj_mean,leakage_rate_mean,recall_at_k_mean,mrr_mean,retrieval_time_mean,generation_time_mean,total_time_mean,n_queries
1,M5_DENSE_OAI_RAG,1,0.0,3500,4.640,0.433333,0.097222,0.266667,0.289620,7.207142,7.496856,30
0,M3_DENSE_ST_RAG,1,0.0,3500,4.415,0.633333,0.075000,0.233333,0.063661,7.224708,7.288462,30


### Interpretación del ajuste fino (tuning reducido)

El tuning reducido exploró combinaciones de:

- k ∈ {1, 3}
- temperature ∈ {0.0, 0.2}
- context_max_chars = 3500

Principales hallazgos:

1. **k = 1 obtuvo consistentemente mejores resultados que k = 3.**
   - Incluir múltiples pasajes incrementó el riesgo de mezclar contexto.
   - Un único pasaje altamente relevante favorece respuestas más enfocadas y reduce leakage.

2. **temperature = 0.0 superó a 0.2.**
   - Mayor determinismo reduce la probabilidad de que el modelo cite referencias fuera del contexto recuperado.
   - Para un sistema RAG pastoral, el control de alucinaciones es más crítico que la creatividad expresiva.

3. **M5 (Dense-RAG con embeddings OpenAI) superó consistentemente a M3 (embeddings locales).**
   - Mejor `judge_overall_adj_mean`
   - Menor leakage_rate
   - Mejor desempeño en métricas IR (Recall@k y MRR)

En validación FULL (30 queries), la mejor configuración fue:

- Modelo: M5_DENSE_OAI_RAG
- k = 1
- temperature = 0.0
- context_max_chars = 3500

Esta configuración logró el mayor equilibrio entre:
- Fidelidad al contexto,
- Reducción de citas fuera de contexto,
- Relevancia pastoral,
- Eficiencia computacional.

El ajuste fino confirma que el valor principal de LUMINA no está en mayor cantidad de contexto ni en creatividad generativa, sino en precisión de recuperación y control estricto del grounding.

---

> *“Porque las cosas invisibles de él, su eterno poder y deidad, se hacen claramente visibles desde la creación del mundo, siendo entendidas por medio de las cosas hechas…”*  
>
> Romanos 1:20

---

# **PARTE 10 – Modelo individual final seleccionado**

## Modelo final elegido

**M5_DENSE_OAI_RAG**
- Retriever: Dense (OpenAI embeddings)
- k = 1
- temperature = 0.0
- context_max_chars = 3500

## Justificación técnica

El modelo fue seleccionado con base en:

- Mejor desempeño en evaluación STRICT (`judge_overall_adj_mean`)
- Menor tasa de citation leakage
- Mejor rendimiento en métricas IR
- Comportamiento determinista y estable
- Tiempos de respuesta razonables

## Trade-offs considerados

1. Dependencia de API externa (OpenAI embeddings)
   - Ventaja: mejor representación semántica.
   - Desventaja: costo y dependencia de servicio.

2. k = 1 vs múltiples pasajes
   - Reduce complejidad contextual.
   - Mejora precisión y coherencia.

3. temperature = 0.0
   - Reduce creatividad.
   - Aumenta confiabilidad y trazabilidad.

In [30]:
FINAL_MODEL_CONFIG = {
    "model_id": "M5_DENSE_OAI_RAG",
    "k": 1,
    "temperature": 0.0,
    "context_max_chars": 3500
}

print("Modelo final seleccionado:")
print(FINAL_MODEL_CONFIG)

Modelo final seleccionado:
{'model_id': 'M5_DENSE_OAI_RAG', 'k': 1, 'temperature': 0.0, 'context_max_chars': 3500}


---

> *"Por lo tanto, ya que estamos recibiendo un reino que no puede ser sacudido, seamos agradecidos. Así podremos servir a Dios de una manera que le agrade, con respeto y temor reverente."*
>
> Hebreos 12:28

---

# **Conclusión general – Avance 4: Modelos Alternativos**

El presente avance tuvo como objetivo explorar múltiples configuraciones del sistema LUMINA para identificar la arquitectura que maximiza desempeño bajo criterios técnicos rigurosos y evaluación trazable.

A diferencia del baseline del Avance 3, en esta etapa:

- Se construyeron 6 modelos individuales (no ensambles).
- Se compararon bajo métricas IR (Recall@k, MRR) y métricas generativas.
- Se implementó una evaluación STRICT con verificación explícita de grounding.
- Se introdujo penalización automática por citation leakage.
- Se realizó ajuste fino controlado sobre los dos mejores modelos.

## Hallazgos estructurales clave

1. **La evaluación naïve sobreestima LLM-only.**
   Bajo evaluación estricta, el modelo sin RAG presentó leakage del 100%, confirmando la necesidad de grounding explícito.

2. **Los modelos RAG superan consistentemente a LLM-only cuando se exige trazabilidad.**
   El valor real de LUMINA no está en mejorar el tono pastoral (donde el LLM ya es fuerte), sino en reducir alucinaciones y controlar citas fuera de contexto.

3. **Dense retrieval supera BM25 e híbrido.**
   Los métodos léxicos mostraron menor recall y mayor leakage relativo.

4. **Embeddings OpenAI superaron a embeddings locales.**
   El modelo M5 (Dense OpenAI) logró:
   - Mayor `judge_overall_adj_mean`
   - Menor leakage
   - Mejor balance entre IR y fidelidad

5. **Menos contexto es mejor (k = 1).**
   Incluir múltiples pasajes incrementa mezcla contextual y riesgo de leakage.
   Un único pasaje altamente relevante produce respuestas más precisas y controladas.

6. **Determinismo > creatividad para un sistema pastoral grounded.**
   `temperature = 0.0` redujo leakage y mejoró consistencia.

## Modelo final seleccionado

**M5_DENSE_OAI_RAG**
- k = 1
- temperature = 0.0
- context_max_chars = 3500

Este modelo ofrece el mejor equilibrio entre:

- Fidelidad bíblica
- Trazabilidad verificable
- Reducción de alucinaciones
- Coherencia pastoral
- Eficiencia operativa

## Reflexión metodológica

Este avance confirma que el diseño de un sistema RAG no debe optimizar únicamente relevancia superficial o calidad retórica, sino control explícito del grounding.

La evaluación estricta permitió evidenciar diferencias que no eran visibles bajo un judge tradicional, fortaleciendo la validez técnica del modelo final.

Con esto, LUMINA avanza hacia una arquitectura estable, reproducible y defendible académicamente.

---

> *“Pregunta ahora a las bestias, y ellas te enseñarán; a las aves de los cielos, y ellas te lo mostrarán… ¿Quién no sabe que la mano de Jehová ha hecho esto?”*
>
> Job 12:7-9

---